[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/06_eigenvalues_eigenvectors_spectral_theory/exercises.ipynb)

# Module 06 — Exercises: Eigenvalues, Eigenvectors, and Spectral Theory

Forty solved problems in four tiers. Every problem carries a statement, a one-line intuition, a
stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is numeric or
algorithmic — a code cell that recomputes it.

Theorem and proof numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): eigenvalues descending, norms written
$\lVert x \rVert$, transition matrices column-stochastic.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
print(f"machine epsilon = {EPS:.4e}")

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Spectrum of the identity

**Statement.** Give the eigenvalues of $I_n$ with both multiplicities.

**Intuition.** The identity scales every direction by the same factor, so every non-zero vector
is an eigenvector.

**Solution.**

*Step 1.* $p_I(z) = \det(I - zI) = (1-z)^n$, whose only root is $z = 1$ with $m_1 = n$.

*Step 2.* $I - 1 \cdot I = 0$, so $\operatorname{Null}(I - I) = \mathbb{R}^n$ and $g_1 = n$.

$$
\boxed{\lambda = 1, \qquad m_1 = g_1 = n}
$$

**Key takeaway.** $g_\lambda = m_\lambda = n$ is the extreme case of Proposition 4.10: the whole
space is one eigenspace.

In [2]:
n = 5
I = np.eye(n)
print("eigenvalues:", np.linalg.eigvalsh(I))
print("geometric multiplicity of 1:", n - np.linalg.matrix_rank(I - np.eye(n)))
assert np.allclose(np.linalg.eigvalsh(I), np.ones(n))
assert n - np.linalg.matrix_rank(I - np.eye(n)) == n

eigenvalues: [1. 1. 1. 1. 1.]
geometric multiplicity of 1: 5


### Problem L0.2 — Spectrum of a diagonal matrix

**Statement.** Give the eigenvalues of $D = \operatorname{diag}(d_1, \dots, d_n)$.

**Intuition.** A diagonal matrix scales each coordinate axis separately, so the axes are the
eigenvectors.

**Solution.**

*Step 1.* $\det(D - zI) = \prod_{i=1}^{n} (d_i - z)$, because the matrix is diagonal.

*Step 2.* The roots are exactly the diagonal entries, and $De_i = d_i e_i$.

$$
\boxed{\lambda_i = d_i \text{ with eigenvector } e_i}
$$

**Key takeaway.** The same argument works for any triangular matrix, since its determinant is
also the product of the diagonal.

In [3]:
d = np.array([4.0, -1.0, 2.5, 0.0])
D = np.diag(d)
print("diagonal entries:", d)
print("eigenvalues     :", np.sort(np.linalg.eigvals(D).real))
U = np.triu(rng.standard_normal((4, 4)))
print("upper triangular: diag", np.sort(np.diag(U)), " eigs", np.sort(np.linalg.eigvals(U).real))
assert np.allclose(np.sort(np.linalg.eigvals(D).real), np.sort(d))
assert np.allclose(np.sort(np.linalg.eigvals(U).real), np.sort(np.diag(U)))

diagonal entries: [ 4.  -1.   2.5  0. ]
eigenvalues     : [-1.   0.   2.5  4. ]
upper triangular: diag [-0.7323 -0.6233  0.1257  0.3616]  eigs [-0.7323 -0.6233  0.1257  0.3616]


### Problem L0.3 — A polynomial of a matrix

**Statement.** If $Av = \lambda v$ with $v \neq 0$, what is the eigenvalue of $v$ for
$P(A) = A^2 + 3A$?

**Intuition.** Every power of $A$ acts on $v$ by the matching power of $\lambda$.

**Solution.**

*Step 1.* $A^2 v = A(\lambda v) = \lambda^2 v$.

*Step 2.* Hence $(A^2 + 3A)v = (\lambda^2 + 3\lambda) v$.

$$
\boxed{\lambda^2 + 3\lambda}
$$

**Key takeaway.** For any polynomial $P$, $P(A)v = P(\lambda)v$ — the identity behind
Cayley-Hamilton reductions and matrix functions.

In [4]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
lam, V = np.linalg.eigh(A)
PA = A @ A + 3 * A
for i in range(2):
    v, l = V[:, i], lam[i]
    print(f"lambda = {l:.4f}   P(lambda) = {l**2 + 3*l:.4f}   "
          f"||P(A)v - P(lambda)v|| = {np.linalg.norm(PA @ v - (l**2 + 3*l) * v):.3e}")
    assert np.linalg.norm(PA @ v - (l ** 2 + 3 * l) * v) < 1e-12

lambda = 1.0000   P(lambda) = 4.0000   ||P(A)v - P(lambda)v|| = 6.280e-16
lambda = 3.0000   P(lambda) = 18.0000   ||P(A)v - P(lambda)v|| = 2.512e-15


### Problem L0.4 — Spectrum of the inverse

**Statement.** Let $A$ be invertible with $Av = \lambda v$, $v \neq 0$. Find the eigenvalue of
$v$ for $A^{-1}$.

**Intuition.** Undoing a stretch by $\lambda$ is a stretch by $1/\lambda$.

**Solution.**

*Step 1.* Invertibility gives $\lambda \neq 0$: a zero eigenvalue would put $v$ in
$\operatorname{Null}(A)$.

*Step 2.* Apply $A^{-1}$ to $Av = \lambda v$: $v = \lambda A^{-1} v$, so
$A^{-1} v = \lambda^{-1} v$.

$$
\boxed{\lambda(A^{-1}) = 1/\lambda(A), \text{ same eigenvector}}
$$

**Key takeaway.** Inversion inverts eigenvalues and leaves eigenvectors alone — the basis of
inverse iteration, which converts a small eigenvalue into a dominant one.

In [5]:
A = np.array([[3.0, 1.0], [1.0, 3.0]])
lam = np.linalg.eigvalsh(A)
lam_inv = np.linalg.eigvalsh(np.linalg.inv(A))
print("eigenvalues of A     :", lam[::-1])
print("eigenvalues of A^-1  :", lam_inv[::-1])
print("reciprocals of the above:", 1 / lam)
assert np.allclose(np.sort(lam_inv), np.sort(1 / lam))

eigenvalues of A     : [4. 2.]
eigenvalues of A^-1  : [0.5  0.25]
reciprocals of the above: [0.5  0.25]


### Problem L0.5 — Transpose has the same spectrum

**Statement.** Show that $A$ and $A^{\top}$ have the same eigenvalues with the same algebraic
multiplicities.

**Intuition.** Transposing does not change a determinant, and the eigenvalues are the roots of a
determinant.

**Solution.**

*Step 1.* $(A - zI)^{\top} = A^{\top} - zI$.

*Step 2.* $\det(M^{\top}) = \det(M)$, so
$p_{A^{\top}}(z) = \det(A^{\top} - zI) = \det(A - zI) = p_A(z)$.

$$
\boxed{p_{A^{\top}} = p_A, \text{ so } \operatorname{spec}(A^{\top}) = \operatorname{spec}(A)}
$$

**Key takeaway.** The eigen*vectors* generally differ: those of $A^{\top}$ are the left
eigenvectors of $A$, which is why a Markov chain's stationary vector depends on whether $P$ is
row- or column-stochastic.

In [6]:
A = rng.standard_normal((4, 4))
print("eigs of A  :", np.sort_complex(np.linalg.eigvals(A)))
print("eigs of A^T:", np.sort_complex(np.linalg.eigvals(A.T)))
print("char polys equal:", np.allclose(np.poly(A), np.poly(A.T)))
_, VA = np.linalg.eig(A)
_, VT = np.linalg.eig(A.T)
print("first eigenvector of A  :", VA[:, 0].real)
print("first eigenvector of A^T:", VT[:, 0].real, " (different in general)")
assert np.allclose(np.sort_complex(np.linalg.eigvals(A)), np.sort_complex(np.linalg.eigvals(A.T)))

eigs of A  : [-1.7471+0.j      0.0657-0.6103j  0.0657+0.6103j  1.4852+0.j    ]
eigs of A^T: [-1.7471+0.j      0.0657-0.6103j  0.0657+0.6103j  1.4852+0.j    ]
char polys equal: True
first eigenvector of A  : [-0.0708  0.9783 -0.0636  0.1841]
first eigenvector of A^T: [ 0.2934  0.0964 -0.702  -0.6417]  (different in general)


### Problem L0.6 — Spectrum of a projection

**Statement.** Show that $P^2 = P$ forces every eigenvalue of $P$ into $\{0, 1\}$.

**Intuition.** Projecting twice is projecting once, so the scale factor must satisfy
$\lambda^2 = \lambda$.

**Solution.**

*Step 1.* From $Pv = \lambda v$, $P^2 v = \lambda^2 v$.

*Step 2.* $P^2 = P$ gives $\lambda^2 v = \lambda v$, and $v \neq 0$ gives
$\lambda(\lambda - 1) = 0$.

$$
\boxed{\lambda \in \{0, 1\}}
$$

**Key takeaway.** The eigenspace for $1$ is the range and the eigenspace for $0$ is the null
space; $\operatorname{tr} P = \operatorname{rank} P$ follows immediately.

In [7]:
a = np.array([1.0, 2.0, -1.0])
P = np.outer(a, a) / (a @ a)
print("||P^2 - P||       =", np.linalg.norm(P @ P - P))
print("eigenvalues       :", np.round(np.linalg.eigvalsh(P), 12))
print("trace = rank      :", np.trace(P), np.linalg.matrix_rank(P))
assert np.allclose(np.sort(np.round(np.linalg.eigvalsh(P), 10)), [0.0, 0.0, 1.0])

||P^2 - P||       = 1.6653345369377348e-16
eigenvalues       : [-0.  0.  1.]
trace = rank      : 0.9999999999999999 1


### Problem L0.7 — Spectrum of an involution

**Statement.** Show that $R^2 = I$ forces every eigenvalue of $R$ into $\{-1, +1\}$.

**Intuition.** Reflecting twice returns you to the start, so the scale factor squares to one.

**Solution.**

*Step 1.* From $Rv = \lambda v$, $R^2 v = \lambda^2 v = v$.

*Step 2.* $v \neq 0$ gives $\lambda^2 = 1$.

$$
\boxed{\lambda \in \{-1, +1\}}
$$

**Key takeaway.** An involution splits the space into a fixed subspace and a flipped one; a
Householder reflector is the case with a single $-1$.

In [8]:
u = np.array([1.0, 1.0, 0.0])
u = u / np.linalg.norm(u)
R = np.eye(3) - 2 * np.outer(u, u)
print("||R^2 - I|| =", np.linalg.norm(R @ R - np.eye(3)))
print("eigenvalues :", np.round(np.linalg.eigvalsh(R), 12))
assert np.allclose(np.sort(np.round(np.linalg.eigvalsh(R), 10)), [-1.0, 1.0, 1.0])

||R^2 - I|| = 8.881784197001251e-16
eigenvalues : [-1.  1.  1.]


### Problem L0.8 — Spectral radius of a two-by-two

**Statement.** Compute $\rho(A)$ for $A = \begin{pmatrix} 0 & 2 \\ 2 & 3 \end{pmatrix}$.

**Intuition.** For a two-by-two the characteristic polynomial is
$z^2 - (\operatorname{tr} A) z + \det A$.

**Solution.**

*Step 1.* $\operatorname{tr} A = 3$ and $\det A = -4$, so $p_A(z) = z^2 - 3z - 4$.

*Step 2.* $z^2 - 3z - 4 = (z-4)(z+1)$, giving $\lambda_1 = 4$, $\lambda_2 = -1$.

*Step 3.* $\rho(A) = \max\{4, 1\} = 4$.

$$
\boxed{\rho(A) = 4}
$$

**Key takeaway.** The spectral radius, not the largest eigenvalue, controls $\lVert A^k \rVert$;
here the negative eigenvalue is irrelevant because its modulus is smaller.

In [9]:
A = np.array([[0.0, 2.0], [2.0, 3.0]])
lam = np.linalg.eigvalsh(A)[::-1]
print("trace, det :", np.trace(A), np.linalg.det(A))
print("eigenvalues:", lam)
print("rho(A)     :", np.abs(lam).max())
assert np.allclose(np.sort(lam), [-1.0, 4.0])
assert abs(np.abs(lam).max() - 4.0) < 1e-12

trace, det : 3.0 -4.0
eigenvalues: [ 4. -1.]
rho(A)     : 4.0


## L1 — Foundations

### Problem L1.1 — A matrix that cannot be diagonalized

**Statement.** Is $A = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}$ diagonalizable? Give both
multiplicities.

**Intuition.** $A$ sends $e_2 \mapsto e_1$ and $e_1 \mapsto 0$: it has only one invariant
direction, not two.

**Solution.**

*Step 1.* $p_A(z) = z^2$, so $\lambda = 0$ with $m_0 = 2$.

*Step 2.* $\operatorname{Null}(A) = \operatorname{span}(e_1)$, since $Ax = 0$ forces $x_2 = 0$.
Hence $g_0 = 1$.

*Step 3.* $g_0 = 1 \lt 2 = m_0$, so by Proposition 4.10 the matrix is defective.

$$
\boxed{\text{Not diagonalizable: } g_0 = 1 \lt m_0 = 2}
$$

**Key takeaway.** Nilpotency and defectiveness are different statements about the same matrix:
$A^2 = 0$ is about powers, $g_0 \lt m_0$ is about eigenvectors.

In [10]:
A = np.array([[0.0, 1.0], [0.0, 0.0]])
print("char poly     :", np.poly(A))
print("eigenvalues   :", np.linalg.eigvals(A))
print("m_0 = 2, g_0  =", 2 - np.linalg.matrix_rank(A))
w, V = np.linalg.eig(A)
print("eigenvector matrix rank:", np.linalg.matrix_rank(V), "(needs 2 to diagonalize)")
assert 2 - np.linalg.matrix_rank(A) == 1
assert np.linalg.matrix_rank(V) == 1

char poly     : [1. 0. 0.]
eigenvalues   : [0. 0.]
m_0 = 2, g_0  = 1
eigenvector matrix rank: 1 (needs 2 to diagonalize)


### Problem L1.2 — Trace and determinant from the spectrum

**Statement.** Express $\operatorname{tr} A$ and $\det A$ in terms of the eigenvalues
$\lambda_1, \dots, \lambda_n$ of $A \in \mathbb{C}^{n \times n}$.

**Intuition.** Both are coefficients of the characteristic polynomial, which factors over
$\mathbb{C}$ into the eigenvalues.

**Solution.**

*Step 1.* Over $\mathbb{C}$, $p_A(z) = \prod_{i=1}^{n} (\lambda_i - z)$.

*Step 2 — determinant.* Evaluate at $z = 0$: $\det A = p_A(0) = \prod_i \lambda_i$.

*Step 3 — trace.* Expand $\det(A - zI)$ by the Leibniz formula,

$$
\det(A - zI) = \sum_{\sigma} \operatorname{sgn}(\sigma) \prod_{i=1}^{n} (A - zI)_{i\,\sigma(i)} .
$$

A permutation $\sigma \neq \operatorname{id}$ moves at least two indices, so its product contains
at most $n-2$ diagonal factors $(a_{ii} - z)$ and has degree at most $n-2$ in $z$. Only the
identity permutation can contribute to the coefficient of $z^{n-1}$, and it contributes
$\prod_i (a_{ii} - z)$, whose $z^{n-1}$ coefficient is $(-1)^{n-1} \sum_i a_{ii}$.

*Step 4.* The same coefficient in $\prod_i (\lambda_i - z)$ is
$(-1)^{n-1} \sum_i \lambda_i$. Equating gives the trace identity.

$$
\boxed{\operatorname{tr} A = \sum_{i=1}^{n} \lambda_i, \qquad \det A = \prod_{i=1}^{n} \lambda_i}
$$

**Key takeaway.** These are the two cheapest checks on any eigenvalue computation, and they hold
for defective matrices too, since both sides count algebraic multiplicity.

In [11]:
A = rng.standard_normal((5, 5))
lam = np.linalg.eigvals(A)
print("trace       :", np.trace(A), "   sum of eigenvalues:", lam.sum().real)
print("determinant :", np.linalg.det(A), "   product of eigenvalues:", np.prod(lam).real)
N = np.array([[0.0, 1.0], [0.0, 0.0]])
print("defective case: trace", np.trace(N), "sum eigs", np.linalg.eigvals(N).sum().real)
assert abs(np.trace(A) - lam.sum().real) < 1e-10
assert abs(np.linalg.det(A) - np.prod(lam).real) < 1e-10

trace       : 1.6764501959573415    sum of eigenvalues: 1.676450195957344
determinant : -3.3012536325850794    product of eigenvalues: -3.3012536325850887
defective case: trace 0.0 sum eigs 0.0


### Problem L1.3 — Eigenpairs of a symmetric two-by-two

**Statement.** Compute the eigenvalues and normalized eigenvectors of
$A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$, and verify orthogonality.

**Intuition.** $A = I + \begin{pmatrix}1&1\\1&1\end{pmatrix}$, and the all-ones block has
eigenvalues $2$ and $0$.

**Solution.**

*Step 1.* $p_A(z) = (2-z)^2 - 1 = z^2 - 4z + 3 = (z-3)(z-1)$, so $\lambda_1 = 3$,
$\lambda_2 = 1$.

*Step 2.* For $\lambda_1 = 3$: $(A - 3I) = \begin{pmatrix}-1&1\\1&-1\end{pmatrix}$, whose null
space is spanned by $(1,1)$.

*Step 3.* For $\lambda_2 = 1$: $(A - I) = \begin{pmatrix}1&1\\1&1\end{pmatrix}$, whose null space
is spanned by $(1,-1)$.

*Step 4.* $(1,1) \cdot (1,-1) = 0$, as Theorem 4.2 requires.

$$
\boxed{\lambda_1 = 3, \; q_1 = \tfrac{1}{\sqrt2}(1,1)^{\top}; \qquad \lambda_2 = 1, \; q_2 = \tfrac{1}{\sqrt2}(1,-1)^{\top}}
$$

**Key takeaway.** This is the matrix drawn in Section 2 of the theory notebook: the ellipse axes
are exactly these two directions with semi-axes $3$ and $1$.

In [12]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
lam, Q = np.linalg.eigh(A)
lam, Q = lam[::-1], Q[:, ::-1]
print("eigenvalues :", lam)
print("eigenvectors:\n", Q)
print("q1 . q2     :", Q[:, 0] @ Q[:, 1])
print("||A - Q L Q^T||:", np.linalg.norm(A - Q @ np.diag(lam) @ Q.T))
assert np.allclose(lam, [3.0, 1.0])
assert abs(Q[:, 0] @ Q[:, 1]) < 1e-14

eigenvalues : [3. 1.]
eigenvectors:
 [[ 0.7071 -0.7071]
 [ 0.7071  0.7071]]
q1 . q2     : 0.0
||A - Q L Q^T||: 3.510833468576701e-16


### Problem L1.4 — Spectrum of a rank-one matrix

**Statement.** Find all eigenvalues of $A = uv^{\top}$ with $u, v \in \mathbb{R}^n$ non-zero,
together with their multiplicities.

**Intuition.** $A$ maps everything onto the single line $\operatorname{span}(u)$, so $n-1$
dimensions are annihilated.

**Solution.**

*Step 1.* $\operatorname{rank} A = 1$, so $\dim \operatorname{Null}(A) = n-1$ and $\lambda = 0$
has $g_0 = n-1$.

*Step 2.* $Au = u(v^{\top} u) = (v^{\top}u) u$, so $u$ is an eigenvector with eigenvalue
$v^{\top} u$.

*Step 3.* $\operatorname{tr} A = v^{\top} u$, and by Problem L1.2 the eigenvalues sum to it, so
the remaining $n-1$ eigenvalues are $0$.

*Step 4.* If $v^{\top} u = 0$ then $\lambda = 0$ has $m_0 = n$ but still $g_0 = n-1$, and $A$ is
defective.

$$
\boxed{\lambda_1 = v^{\top} u \; (m = 1), \qquad \lambda = 0 \; (m = n-1)}
$$

**Key takeaway.** Rank-one updates appear everywhere — power iteration deflation, BFGS,
attention scores — and their spectrum is a single inner product.

In [13]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, -1.0, 1.0])
A = np.outer(u, v)
ev = np.sort(np.linalg.eigvals(A).real)[::-1]
print("v^T u        :", v @ u)
print("eigenvalues  :", np.round(ev, 12))
print("rank(A)      :", np.linalg.matrix_rank(A), "  dim Null(A) =", 3 - np.linalg.matrix_rank(A))

w = np.array([1.0, 1.0, -1.0])            # w^T u = 1 + 2 - 3 = 0
Ad = np.outer(u, w)
print("\ndegenerate case w^T u = 0")
print("  w^T u                    :", w @ u)
print("  eigenvalues              :", np.round(np.sort(np.linalg.eigvals(Ad).real), 6))
print("  m_0 = 3, g_0             :", 3 - np.linalg.matrix_rank(Ad), "-> defective")
print("  cond(V) of eigenvectors  : {:.3e}".format(np.linalg.cond(np.linalg.eig(Ad)[1])))
assert abs(ev[0] - v @ u) < 1e-10
assert np.allclose(ev[1:], 0.0, atol=1e-10)
assert 3 - np.linalg.matrix_rank(Ad) == 2

v^T u        : 5.0
eigenvalues  : [ 5.  0. -0.]
rank(A)      : 1   dim Null(A) = 2

degenerate case w^T u = 0
  w^T u                    : 0.0
  eigenvalues              : [-0.  0.  0.]
  m_0 = 3, g_0             : 2 -> defective
  cond(V) of eigenvectors  : 1.656e+08


### Problem L1.5 — A real matrix with no real eigenvalue

**Statement.** Find the eigenvalues of
$R_\theta = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$
for $\theta \in (0, \pi)$.

**Intuition.** A genuine rotation of the plane fixes no line, so no real eigenvector can exist.

**Solution.**

*Step 1.* $p(z) = (\cos\theta - z)^2 + \sin^2\theta = z^2 - 2z\cos\theta + 1$.

*Step 2.* The discriminant is $4\cos^2\theta - 4 = -4\sin^2\theta \lt 0$ for
$\theta \in (0,\pi)$, so

$$
z = \cos\theta \pm i \sin\theta = e^{\pm i\theta}.
$$

$$
\boxed{\lambda = e^{i\theta}, \; e^{-i\theta}}
$$

**Key takeaway.** This is why Theorem 4.1 is stated over $\mathbb{C}$: over $\mathbb{R}$ there is
no triangularizing basis at all.

In [14]:
for theta in (np.pi / 6, np.pi / 2, 2 * np.pi / 3):
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    ev = np.linalg.eigvals(R)
    print(f"theta = {theta:.4f}   eigenvalues = {np.round(ev, 6)}   "
          f"exp(+-i theta) = {np.round([np.exp(1j*theta), np.exp(-1j*theta)], 6)}")
    assert np.allclose(np.sort_complex(ev), np.sort_complex(np.array([np.exp(1j * theta), np.exp(-1j * theta)])))

theta = 0.5236   eigenvalues = [0.866+0.5j 0.866-0.5j]   exp(+-i theta) = [0.866+0.5j 0.866-0.5j]
theta = 1.5708   eigenvalues = [0.+1.j 0.-1.j]   exp(+-i theta) = [0.+1.j 0.-1.j]
theta = 2.0944   eigenvalues = [-0.5+0.866j -0.5-0.866j]   exp(+-i theta) = [-0.5+0.866j -0.5-0.866j]


### Problem L1.6 — A skew-symmetric two-by-two

**Statement.** Find the eigenvalues of
$S = \begin{pmatrix} 0 & -\omega \\ \omega & 0 \end{pmatrix}$.

**Intuition.** $S$ is the generator of rotation at angular velocity $\omega$, so its spectrum
should be purely imaginary.

**Solution.**

*Step 1.* $p_S(z) = z^2 + \omega^2$.

*Step 2.* The roots are $z = \pm i\omega$.

$$
\boxed{\lambda = i\omega, \; -i\omega}
$$

**Key takeaway.** $S$ is normal but not symmetric, so Theorem 4.3 applies and Theorem 4.2 does
not: the eigenvectors are orthogonal in $\mathbb{C}^2$, but the eigenvalues are not real.

In [15]:
omega = 1.7
S = np.array([[0.0, -omega], [omega, 0.0]])
ev, Vc = np.linalg.eig(S)
print("eigenvalues      :", np.round(ev, 8))
print("S is normal      :", np.allclose(S.conj().T @ S, S @ S.conj().T))
print("S is symmetric   :", np.allclose(S, S.T))
print("|q1* q2| (complex):", abs(Vc[:, 0].conj() @ Vc[:, 1]))
assert np.allclose(np.sort_complex(ev), np.sort_complex(np.array([1j * omega, -1j * omega])))
assert np.allclose(S.conj().T @ S, S @ S.conj().T)

eigenvalues      : [0.+1.7j 0.-1.7j]
S is normal      : True
S is symmetric   : False
|q1* q2| (complex): 0.0


### Problem L1.7 — Algebraic against geometric multiplicity

**Statement.** Give both multiplicities for every eigenvalue of
$A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 3 & 0 \\ 0 & 0 & 4 \end{pmatrix}$.

**Intuition.** The upper-left block is a Jordan block: two copies of the eigenvalue $3$ but only
one eigenvector.

**Solution.**

*Step 1.* $A$ is triangular, so $p_A(z) = (3-z)^2 (4-z)$, giving $m_3 = 2$ and $m_4 = 1$.

*Step 2.* $A - 3I = \begin{pmatrix} 0&1&0 \\ 0&0&0 \\ 0&0&1 \end{pmatrix}$ has rank $2$, so
$g_3 = 3 - 2 = 1$.

*Step 3.* $A - 4I$ has rank $2$, so $g_4 = 1 = m_4$.

$$
\boxed{m_3 = 2,\; g_3 = 1; \qquad m_4 = 1,\; g_4 = 1}
$$

**Key takeaway.** One deficient eigenvalue is enough to make the whole matrix defective, whatever
the other eigenvalues do.

In [16]:
A = np.array([[3.0, 1, 0], [0, 3, 0], [0, 0, 4]])
print("char poly:", np.poly(A))
for l, m in [(3.0, 2), (4.0, 1)]:
    g = 3 - np.linalg.matrix_rank(A - l * np.eye(3))
    print(f"  lambda = {l}:  m = {m}, g = {g}")
    assert g == 1
print("rank of eigenvector basis:", np.linalg.matrix_rank(np.linalg.eig(A)[1]), "(needs 3)")
assert np.linalg.matrix_rank(np.linalg.eig(A)[1]) < 3

char poly: [  1. -10.  33. -36.]
  lambda = 3.0:  m = 2, g = 1
  lambda = 4.0:  m = 1, g = 1
rank of eigenvector basis: 2 (needs 3)


### Problem L1.8 — Spectral decomposition into projectors

**Statement.** Write $A = \begin{pmatrix} 5 & 2 \\ 2 & 2 \end{pmatrix}$ as
$\sum_i \lambda_i q_i q_i^{\top}$.

**Intuition.** Theorem 4.2 splits a symmetric matrix into orthogonal rank-one channels, each
scaled by its own eigenvalue.

**Solution.**

*Step 1.* $p_A(z) = z^2 - 7z + 6 = (z-6)(z-1)$, so $\lambda_1 = 6$, $\lambda_2 = 1$.

*Step 2.* $(A - 6I) = \begin{pmatrix}-1&2\\2&-4\end{pmatrix}$ has null space spanned by $(2,1)$,
so $q_1 = \tfrac{1}{\sqrt5}(2,1)^{\top}$.

*Step 3.* $(A - I) = \begin{pmatrix}4&2\\2&1\end{pmatrix}$ has null space spanned by $(-1,2)$,
so $q_2 = \tfrac{1}{\sqrt5}(-1,2)^{\top}$.

*Step 4.* Therefore

$$
q_1 q_1^{\top} = \frac{1}{5}\begin{pmatrix}4&2\\2&1\end{pmatrix},
\qquad
q_2 q_2^{\top} = \frac{1}{5}\begin{pmatrix}1&-2\\-2&4\end{pmatrix}.
$$

$$
\boxed{A = 6 \cdot \frac{1}{5}\begin{pmatrix}4&2\\2&1\end{pmatrix} + 1 \cdot \frac{1}{5}\begin{pmatrix}1&-2\\-2&4\end{pmatrix}}
$$

**Key takeaway.** The projectors satisfy $P_1 + P_2 = I$ and $P_1 P_2 = 0$, so any function of
$A$ is $f(6)P_1 + f(1)P_2$ — the fastest route to $A^{1/2}$, $A^{-1}$ or $e^A$.

In [17]:
A = np.array([[5.0, 2.0], [2.0, 2.0]])
q1 = np.array([2.0, 1.0]) / np.sqrt(5)
q2 = np.array([-1.0, 2.0]) / np.sqrt(5)
P1, P2 = np.outer(q1, q1), np.outer(q2, q2)
print("6 P1 + 1 P2 =\n", 6 * P1 + 1 * P2)
print("P1 + P2 = I :", np.allclose(P1 + P2, np.eye(2)))
print("P1 P2 = 0   :", np.allclose(P1 @ P2, np.zeros((2, 2))))
sqrtA = np.sqrt(6) * P1 + np.sqrt(1) * P2
print("sqrt(A)^2 - A residual:", np.linalg.norm(sqrtA @ sqrtA - A))
assert np.allclose(6 * P1 + P2, A)
assert np.linalg.norm(sqrtA @ sqrtA - A) < 1e-12

6 P1 + 1 P2 =
 [[5. 2.]
 [2. 2.]]
P1 + P2 = I : True
P1 P2 = 0   : True
sqrt(A)^2 - A residual: 2.059156955057807e-15


### Problem L1.9 — Localizing a spectrum with Gershgorin

**Statement.** Bound all eigenvalues of
$A = \begin{pmatrix} 4 & 1 & 0 \\ 1 & 3 & 1 \\ 0 & 1 & 2 \end{pmatrix}$ using Theorem 4.8, and
compare with the exact spectrum.

**Intuition.** Each row contributes a disc centred on its diagonal entry with radius the sum of
the other entries in that row.

**Solution.**

*Step 1.* Row $1$: centre $4$, radius $1$, disc $[3,5]$ on the real axis.

*Step 2.* Row $2$: centre $3$, radius $2$, disc $[1,5]$.

*Step 3.* Row $3$: centre $2$, radius $1$, disc $[1,3]$.

*Step 4.* $A$ is symmetric so its eigenvalues are real, and the union of the discs is $[1,5]$.

*Step 5.* Exactly, $p_A(z) = z^3 - 9z^2 + 24z - 18 = (z-3)(z^2 - 6z + 6)$, so the spectrum is
$\{3 + \sqrt3, \, 3, \, 3 - \sqrt3\} \approx \{4.732, 3, 1.268\} \subset [1,5]$.

$$
\boxed{\operatorname{spec}(A) \subset [1, 5], \quad \text{exactly } \{3 \pm \sqrt3, \, 3\}}
$$

**Key takeaway.** The bound costs three additions and is not tight here, but it is enough to
prove $A \succ 0$ without computing a single eigenvalue.

In [18]:
A = np.array([[4.0, 1, 0], [1, 3, 1], [0, 1, 2]])
centres = np.diag(A)
radii = np.abs(A).sum(axis=1) - np.abs(centres)
lam = np.linalg.eigvalsh(A)[::-1]
print("centres, radii :", centres, radii)
print("disc union     : [", (centres - radii).min(), ",", (centres + radii).max(), "]")
print("exact spectrum :", lam, "   3 +/- sqrt 3 =", 3 + np.sqrt(3), 3 - np.sqrt(3))
print("char poly      :", np.poly(A))
assert np.allclose(lam, [3 + np.sqrt(3), 3.0, 3 - np.sqrt(3)])
assert lam.min() >= (centres - radii).min() - 1e-12

centres, radii : [4. 3. 2.] [1. 2. 1.]
disc union     : [ 1.0 , 5.0 ]
exact spectrum : [4.7321 3.     1.2679]    3 +/- sqrt 3 = 4.732050807568877 1.2679491924311228
char poly      : [  1.  -9.  24. -18.]


### Problem L1.10 — Cayley-Hamilton power reduction

**Statement.** For $A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$, express $A^3$ as
$\alpha A + \beta I$.

**Intuition.** Cayley-Hamilton says $A$ satisfies its own characteristic polynomial, so every
power beyond $n-1$ collapses.

**Solution.**

*Step 1.* $\operatorname{tr} A = 5$, $\det A = -2$, so $p_A(z) = z^2 - 5z - 2$.

*Step 2.* Cayley-Hamilton gives $A^2 = 5A + 2I$.

*Step 3.* Multiply by $A$: $A^3 = 5A^2 + 2A = 5(5A + 2I) + 2A = 27A + 10I$.

$$
\boxed{A^3 = 27A + 10 I}
$$

**Key takeaway.** Any $f(A)$ for an $n \times n$ matrix reduces to a polynomial of degree at most
$n-1$; this is what makes matrix functions computable without a full series.

In [19]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
print("char poly  :", np.poly(A))
print("A^2        :\n", np.linalg.matrix_power(A, 2))
print("5A + 2I    :\n", 5 * A + 2 * np.eye(2))
print("A^3        :\n", np.linalg.matrix_power(A, 3))
print("27A + 10I  :\n", 27 * A + 10 * np.eye(2))
assert np.allclose(np.linalg.matrix_power(A, 3), 27 * A + 10 * np.eye(2))

char poly  : [ 1. -5. -2.]
A^2        :
 [[ 7. 10.]
 [15. 22.]]
5A + 2I    :
 [[ 7. 10.]
 [15. 22.]]
A^3        :
 [[ 37.  54.]
 [ 81. 118.]]
27A + 10I  :
 [[ 37.  54.]
 [ 81. 118.]]


### Problem L1.11 — A quadratic matrix equation

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ satisfy $A^2 = A + I$. Show $A$ is invertible,
find $A^{-1}$, and determine the possible eigenvalues.

**Intuition.** The equation is the golden-ratio recurrence written for matrices.

**Solution.**

*Step 1.* Rearranging, $A(A - I) = I$, so $A$ is invertible with $A^{-1} = A - I$.

*Step 2.* If $Av = \lambda v$ with $v \neq 0$, then $(A^2 - A - I)v = (\lambda^2 - \lambda - 1)v = 0$,
so $\lambda^2 - \lambda - 1 = 0$.

*Step 3.* $\lambda = \tfrac{1 \pm \sqrt5}{2}$, the golden ratio $\varphi$ and $-1/\varphi$.

$$
\boxed{A^{-1} = A - I, \qquad \lambda \in \left\{ \tfrac{1+\sqrt5}{2}, \; \tfrac{1-\sqrt5}{2} \right\}}
$$

**Key takeaway.** A polynomial identity satisfied by $A$ confines the spectrum to that
polynomial's roots — the tool behind Problems L0.6 and L0.7 as well.

In [20]:
F = np.array([[1.0, 1.0], [1.0, 0.0]])          # the Fibonacci matrix
print("F^2 - F - I  :\n", F @ F - F - np.eye(2))
print("F^-1         :\n", np.linalg.inv(F))
print("F - I        :\n", F - np.eye(2))
print("eigenvalues  :", np.linalg.eigvalsh(F)[::-1])
print("golden ratio :", (1 + np.sqrt(5)) / 2, (1 - np.sqrt(5)) / 2)
assert np.allclose(F @ F, F + np.eye(2))
assert np.allclose(np.linalg.inv(F), F - np.eye(2))
assert np.allclose(np.sort(np.linalg.eigvalsh(F)), np.sort([(1 + np.sqrt(5)) / 2, (1 - np.sqrt(5)) / 2]))

F^2 - F - I  :
 [[0. 0.]
 [0. 0.]]
F^-1         :
 [[ 0.  1.]
 [ 1. -1.]]
F - I        :
 [[ 0.  1.]
 [ 1. -1.]]
eigenvalues  : [ 1.618 -0.618]
golden ratio : 1.618033988749895 -0.6180339887498949


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Principal components of a covariance matrix

**Statement.** For the covariance $C = \begin{pmatrix} 3 & 1 \\ 1 & 3 \end{pmatrix}$, find the
first principal direction and the fraction of variance it explains.

**Intuition.** The first principal component is the direction maximizing the Rayleigh quotient of
$C$, which by Theorem 4.4 is the top eigenvector.

**Solution.**

*Step 1.* $p_C(z) = (3-z)^2 - 1 = (z-4)(z-2)$, so $\lambda_1 = 4$, $\lambda_2 = 2$.

*Step 2.* $(C - 4I) = \begin{pmatrix}-1&1\\1&-1\end{pmatrix}$ has null space spanned by $(1,1)$,
so $q_1 = \tfrac{1}{\sqrt2}(1,1)^{\top}$.

*Step 3.* Total variance is $\operatorname{tr} C = 6$, so the fraction explained is
$\lambda_1 / \operatorname{tr} C = 4/6$.

$$
\boxed{q_1 = \tfrac{1}{\sqrt2}(1,1)^{\top}, \qquad \text{variance explained} = \tfrac23 \approx 66.67\%}
$$

**Key takeaway.** PCA is Theorem 4.2 applied to a covariance matrix; the eigenvalues *are* the
variances along the principal directions, which is why they are non-negative.

In [21]:
C = np.array([[3.0, 1.0], [1.0, 3.0]])
lam, Q = np.linalg.eigh(C)
lam, Q = lam[::-1], Q[:, ::-1]
print("eigenvalues        :", lam)
print("first PC           :", Q[:, 0], "  (up to sign, (1,1)/sqrt2 =", 1 / np.sqrt(2), ")")
print("variance explained :", lam[0] / lam.sum())
X = (np.linalg.cholesky(C) @ rng.standard_normal((2, 50000))).T
proj_var = np.var(X @ Q[:, 0])
print("empirical variance along q1:", proj_var, " vs lambda_1 =", lam[0])
assert np.allclose(lam, [4.0, 2.0])
assert abs(lam[0] / lam.sum() - 2 / 3) < 1e-12
assert abs(proj_var - 4.0) < 0.1

eigenvalues        : [4. 2.]
first PC           : [0.7071 0.7071]   (up to sign, (1,1)/sqrt2 = 0.7071067811865475 )
variance explained : 0.6666666666666666
empirical variance along q1: 4.0127380361607035  vs lambda_1 = 4.0


### Problem L2.2 — Stationary distribution of a Markov chain

**Statement.** Find the stationary distribution of the column-stochastic transition matrix
$P = \begin{pmatrix} 0.8 & 0.3 \\ 0.2 & 0.7 \end{pmatrix}$, and give the convergence rate.

**Intuition.** Stationarity is the eigenvector equation $P\pi = \pi$; everything else in the
spectrum decays.

**Solution.**

*Step 1.* Solve $(P - I)\pi = 0$:
$\begin{pmatrix}-0.2 & 0.3 \\ 0.2 & -0.3\end{pmatrix}\pi = 0$ gives $\pi_1 = \tfrac32 \pi_2$.

*Step 2.* Normalizing with $\pi_1 + \pi_2 = 1$: $\tfrac52 \pi_2 = 1$, so $\pi_2 = 0.4$ and
$\pi_1 = 0.6$.

*Step 3.* $\operatorname{tr} P = 1.5$ and $1$ is an eigenvalue, so the other eigenvalue is
$\lambda_2 = 0.5$; by Problem L2.3 the distance to $\pi$ contracts by $0.5$ per step.

$$
\boxed{\pi = (0.6, \; 0.4)^{\top}, \qquad \text{error contracts by } \lvert \lambda_2 \rvert = 0.5 \text{ per step}}
$$

**Key takeaway.** With the column convention of the notation register, $\pi$ is a genuine right
eigenvector of $P$; with a row-stochastic matrix it would be a left eigenvector instead.

In [22]:
P = np.array([[0.8, 0.3], [0.2, 0.7]])
assert np.allclose(P.sum(axis=0), 1.0)
ev, V = np.linalg.eig(P)
pi = np.real(V[:, int(np.argmin(np.abs(ev - 1.0)))])
pi = pi / pi.sum()
print("eigenvalues :", np.real_if_close(ev))
print("pi          :", pi, "   ||P pi - pi|| =", np.linalg.norm(P @ pi - pi))
x = np.array([1.0, 0.0])
prev = np.linalg.norm(x - pi)
for k in range(1, 6):
    x = P @ x
    err = np.linalg.norm(x - pi)
    print(f"  k={k}  x = {x}   error = {err:.6e}   ratio = {err / prev:.6f}")
    prev = err
assert np.allclose(pi, [0.6, 0.4])
assert abs(sorted(np.real(ev))[0] - 0.5) < 1e-12

eigenvalues : [1.  0.5]
pi          : [0.6 0.4]    ||P pi - pi|| = 0.0
  k=1  x = [0.8 0.2]   error = 2.828427e-01   ratio = 0.500000
  k=2  x = [0.7 0.3]   error = 1.414214e-01   ratio = 0.500000
  k=3  x = [0.65 0.35]   error = 7.071068e-02   ratio = 0.500000
  k=4  x = [0.625 0.375]   error = 3.535534e-02   ratio = 0.500000
  k=5  x = [0.6125 0.3875]   error = 1.767767e-02   ratio = 0.500000


### Problem L2.3 — Convergence rate of power iteration

**Statement.** Let $A$ be symmetric positive semidefinite with
$\lambda_1 \gt \lambda_2 \ge \dots \ge \lambda_n \ge 0$ and let
$x_{k+1} = Ax_k / \lVert A x_k \rVert$ with $q_1^{\top} x_0 \neq 0$. Prove the eigenvector error
contracts at rate $\lambda_2/\lambda_1$.

**Intuition.** Every eigen-component is multiplied by its own eigenvalue each step, so the
dominant one wins geometrically.

**Solution.**

*Step 1.* Write $x_0 = \sum_i c_i q_i$ with $c_1 \neq 0$. Since normalizing does not change
direction, $x_k$ spans the same line as $A^k x_0 = \sum_i c_i \lambda_i^k q_i$.

*Step 2.* The component along $q_1$ is $c_1 \lambda_1^k$ and the orthogonal part has norm
$\bigl(\sum_{i \ge 2} c_i^2 \lambda_i^{2k}\bigr)^{1/2}$, so

$$
\tan \theta(x_k, q_1) = \frac{\bigl( \sum_{i \ge 2} c_i^2 \lambda_i^{2k} \bigr)^{1/2}}{\lvert c_1 \rvert \lambda_1^{k}} .
$$

*Step 3.* Bounding $\lambda_i \le \lambda_2$ for $i \ge 2$ gives

$$
\tan \theta(x_k, q_1) \le \left( \frac{\lambda_2}{\lambda_1} \right)^{k} \tan \theta(x_0, q_1),
$$

and if $c_2 \neq 0$ the ratio of consecutive terms tends to $\lambda_2/\lambda_1$, so the rate is
sharp.

$$
\boxed{\tan \theta(x_k, q_1) \le \left( \lambda_2 / \lambda_1 \right)^{k} \tan \theta(x_0, q_1)}
$$

**Key takeaway.** A small spectral gap is the enemy: at $\lambda_2/\lambda_1 = 0.99$ it takes
about $2300$ iterations to gain ten digits, which is why practical solvers shift and invert.

In [23]:
spec = np.array([5.0, 4.0, 1.0, 0.2])     # ratio 0.8
Qr, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A = Qr @ np.diag(spec) @ Qr.T
A = (A + A.T) / 2
q1 = np.linalg.eigh(A)[1][:, -1]
x = rng.standard_normal(4)
x /= np.linalg.norm(x)
tans = []
for k in range(31):
    c = abs(q1 @ x)
    tans.append(np.sqrt(max(0.0, 1 - c * c)) / c)
    x = A @ x
    x /= np.linalg.norm(x)
tans = np.array(tans)
print("predicted rate:", spec[1] / spec[0])
print("observed ratios (k = 25..30):", np.round(tans[26:31] / tans[25:30], 6))
print("bound holds at every step:", np.all(tans <= tans[0] * (spec[1] / spec[0]) ** np.arange(31) + 1e-12))
print("iterations for 1e-10 at ratio 0.99:", int(np.ceil(np.log(1e-10) / np.log(0.99))))
assert np.all(tans <= tans[0] * (spec[1] / spec[0]) ** np.arange(31) + 1e-12)
assert abs(tans[30] / tans[29] - 0.8) < 1e-3

predicted rate: 0.8
observed ratios (k = 25..30): [0.8 0.8 0.8 0.8 0.8]
bound holds at every step: True
iterations for 1e-10 at ratio 0.99: 2292


### Problem L2.4 — PageRank: why damping controls the convergence rate

**Statement.** Let $P$ be column-stochastic with non-negative entries and let

$$
G = \alpha P + \frac{1 - \alpha}{n} \mathbf{1} \mathbf{1}^{\top}, \qquad 0 \lt \alpha \lt 1 .
$$

Prove that every eigenvalue of $G$ is either $1$ or has modulus at most $\alpha$.

**Intuition.** The damping term acts only on the direction $\mathbf{1}$; on everything orthogonal
to $\mathbf{1}$ in the dual sense, $G$ is literally $\alpha P$.

**Solution.**

*Step 1 — $\rho(P) \le 1$.* For any $x$,

$$
\lVert P x \rVert_1 = \sum_i \Bigl\lvert \sum_j P_{ij} x_j \Bigr\rvert \le \sum_j \Bigl( \sum_i P_{ij} \Bigr) \lvert x_j \rvert = \lVert x \rVert_1 ,
$$

using $P_{ij} \ge 0$ and unit column sums. Hence $\rho(P) \le \lVert P \rVert_1 \le 1$.

*Step 2 — $G$ is column-stochastic.* Each column of $G$ sums to
$\alpha \cdot 1 + (1-\alpha) = 1$, so $\mathbf{1}^{\top} G = \mathbf{1}^{\top}$.

*Step 3 — split on $\mathbf{1}^{\top} x$.* Let $Gx = \lambda x$ with $x \neq 0$.

If $\mathbf{1}^{\top} x \neq 0$, apply $\mathbf{1}^{\top}$ to both sides:
$\mathbf{1}^{\top} x = \mathbf{1}^{\top} G x = \lambda \mathbf{1}^{\top} x$, so $\lambda = 1$.

If $\mathbf{1}^{\top} x = 0$, the rank-one term dies and $Gx = \alpha P x$, so
$Px = (\lambda/\alpha) x$ and $\lvert \lambda \rvert / \alpha \le \rho(P) \le 1$ by Step 1.

$$
\boxed{\operatorname{spec}(G) \subseteq \{1\} \cup \{ z : \lvert z \rvert \le \alpha \}}
$$

**Key takeaway.** Power iteration on $G$ therefore converges at rate $\alpha$ regardless of the
web graph: at Google's $\alpha = 0.85$, ten digits of accuracy need about
$\ln(10^{-10}) / \ln(0.85) \approx 142$ multiplications.

In [24]:
n, alpha = 6, 0.85
Praw = rng.random((n, n))
P = Praw / Praw.sum(axis=0)
G = alpha * P + (1 - alpha) * np.ones((n, n)) / n
assert np.allclose(G.sum(axis=0), 1.0)
evG = np.linalg.eigvals(G)
evG = evG[np.argsort(-np.abs(evG))]
evP = np.linalg.eigvals(P)
evP = evP[np.argsort(-np.abs(evP))]
print("|eig(G)| sorted :", np.round(np.abs(evG), 6))
print("alpha           :", alpha)
print("|lambda_2(G)|   :", abs(evG[1]), "   alpha * |lambda_2(P)| =", alpha * abs(evP[1]))
print("all non-Perron eigenvalues within alpha:", np.all(np.abs(evG[1:]) <= alpha + 1e-12))
print("iterations for 1e-10 :", int(np.ceil(np.log(1e-10) / np.log(alpha))))
assert np.all(np.abs(evG[1:]) <= alpha + 1e-12)
assert abs(abs(evG[1]) - alpha * abs(evP[1])) < 1e-10

|eig(G)| sorted : [1.     0.272  0.1987 0.1987 0.0674 0.0674]
alpha           : 0.85
|lambda_2(G)|   : 0.27203321612762965    alpha * |lambda_2(P)| = 0.2720332161276297
all non-Perron eigenvalues within alpha: True
iterations for 1e-10 : 142


### Problem L2.5 — Polynomial roots as an eigenvalue problem

**Statement.** Build the companion matrix of $p(z) = z^3 - 6z^2 + 11z - 6$ and show its
eigenvalues are the roots of $p$.

**Intuition.** Multiplication by $z$ modulo $p$ is a linear map; its matrix in the basis
$1, z, z^2$ is the companion matrix, and its eigenvalues are the roots.

**Solution.**

*Step 1.* Write $p(z) = z^3 + a_2 z^2 + a_1 z + a_0$ with $a_2 = -6$, $a_1 = 11$, $a_0 = -6$.

*Step 2.* The companion matrix is

$$
C = \begin{pmatrix} 0 & 0 & -a_0 \\ 1 & 0 & -a_1 \\ 0 & 1 & -a_2 \end{pmatrix}
  = \begin{pmatrix} 0 & 0 & 6 \\ 1 & 0 & -11 \\ 0 & 1 & 6 \end{pmatrix}.
$$

*Step 3.* Expanding $\det(zI - C)$ along the first row returns $p(z)$ exactly.

*Step 4.* $p(z) = (z-1)(z-2)(z-3)$, so the spectrum is $\{1,2,3\}$.

$$
\boxed{\operatorname{spec}(C) = \{1, 2, 3\} = \text{roots of } p}
$$

**Key takeaway.** This is not a curiosity: `numpy.roots` builds exactly this matrix and calls the
QR eigenvalue algorithm on it, so root finding and eigenvalue finding are the same computation.

In [25]:
coeffs = [1.0, -6.0, 11.0, -6.0]
C = np.array([[0.0, 0.0, 6.0], [1.0, 0.0, -11.0], [0.0, 1.0, 6.0]])
print("char poly of C :", np.poly(C))
print("eigenvalues    :", np.sort(np.linalg.eigvals(C).real))
print("numpy.roots    :", np.sort(np.roots(coeffs).real))
assert np.allclose(np.poly(C), coeffs)
assert np.allclose(np.sort(np.linalg.eigvals(C).real), [1.0, 2.0, 3.0])
assert np.allclose(np.sort(np.roots(coeffs).real), [1.0, 2.0, 3.0])

char poly of C : [ 1. -6. 11. -6.]
eigenvalues    : [1. 2. 3.]
numpy.roots    : [1. 2. 3.]


### Problem L2.6 — Algebraic connectivity detects connectedness

**Statement.** For the Laplacian $L = D - W$ of a simple undirected graph $G$ on $n$ vertices,
prove that the second smallest eigenvalue is positive if and only if $G$ is connected.

> **Ordering callout.** Laplacian spectra are indexed **ascending** in this repository,
> $0 = \lambda_1 \le \lambda_2 \le \cdots \le \lambda_n$, the reverse of the default. So
> $\lambda_2$ here means the *second smallest* eigenvalue.

**Intuition.** The Laplacian quadratic form measures disagreement across edges; it vanishes only
on functions that are constant on each connected piece.

**Solution.**

*Step 1.* For every $x$, $x^{\top} L x = \sum_{(i,j) \in E} (x_i - x_j)^2 \ge 0$, so
$L \succeq 0$, and $L\mathbf{1} = 0$ gives $\lambda_1 = 0$.

*Step 2 — disconnected implies $\lambda_2 = 0$.* If $G$ has components $V_1, \dots, V_c$ with
$c \ge 2$, the indicator vectors $\mathbf{1}_{V_1}, \dots, \mathbf{1}_{V_c}$ are linearly
independent and each satisfies $L \mathbf{1}_{V_r} = 0$. So $g_0 \ge c \ge 2$ and
$\lambda_2 = 0$.

*Step 3 — connected implies $\lambda_2 \gt 0$.* By Theorem 4.4,

$$
\lambda_2 = \min_{x \perp \mathbf{1}, \; x \neq 0} \frac{\sum_{(i,j) \in E} (x_i - x_j)^2}{\lVert x \rVert^2}.
$$

If the minimum were $0$, the minimizer $x$ would satisfy $x_i = x_j$ along every edge, hence be
constant on the whole graph by connectedness, hence a multiple of $\mathbf{1}$. Combined with
$x \perp \mathbf{1}$ that forces $x = 0$, a contradiction.

$$
\boxed{\lambda_2(L) \gt 0 \iff G \text{ is connected}, \quad \text{and } g_0 = \#\text{components}}
$$

**Key takeaway.** The multiplicity of the eigenvalue $0$ counts connected components exactly,
which is how spectral clustering decides the number of clusters.

In [26]:
def laplacian(edges, n):
    W = np.zeros((n, n))
    for i, j in edges:
        W[i, j] = W[j, i] = 1.0
    return np.diag(W.sum(axis=1)) - W

path = laplacian([(0, 1), (1, 2), (2, 3)], 4)
split = laplacian([(0, 1), (2, 3)], 4)
for name, L in [("connected path", path), ("two components", split)]:
    lam = np.linalg.eigvalsh(L)
    print(f"{name:16s} spectrum {np.round(lam, 6)}   lambda_2 = {lam[1]:.6f}"
          f"   zero multiplicity = {int(np.sum(np.abs(lam) < 1e-10))}")
assert np.linalg.eigvalsh(path)[1] > 1e-8
assert abs(np.linalg.eigvalsh(split)[1]) < 1e-10
assert int(np.sum(np.abs(np.linalg.eigvalsh(split)) < 1e-10)) == 2

connected path   spectrum [0.     0.5858 2.     3.4142]   lambda_2 = 0.585786   zero multiplicity = 1
two components   spectrum [0. 0. 2. 2.]   lambda_2 = 0.000000   zero multiplicity = 2


### Problem L2.7 — Spectral gap of a cycle and diffusive mixing

**Statement.** For the cycle graph $C_n$ with adjacency matrix $A$ (so $d = 2$), compute all
eigenvalues, the second largest one $\lambda_2(A)$, and the spectral gap
$\gamma = d - \lambda_2(A)$.

**Intuition.** $A$ is circulant, so the discrete Fourier modes diagonalize it.

**Solution.**

*Step 1.* Let $\omega = e^{2\pi i/n}$ and $v^{(k)} = (1, \omega^k, \omega^{2k}, \dots, \omega^{(n-1)k})^{\top}$.
Then $(A v^{(k)})_j = \omega^{(j+1)k} + \omega^{(j-1)k} = (\omega^k + \omega^{-k}) v^{(k)}_j$.

*Step 2.* Writing $\lambda^{(k)}$ for the eigenvalue of the $k$-th Fourier mode,

$$
\lambda^{(k)} = \omega^{k} + \omega^{-k} = 2\cos\!\left( \frac{2\pi k}{n} \right), \qquad k = 0, 1, \dots, n-1 .
$$

*Step 3.* The largest is $\lambda^{(0)} = 2 = d$; the next largest are
$\lambda^{(1)} = \lambda^{(n-1)} = 2\cos(2\pi/n)$. Sorting descending, the second largest
eigenvalue is $\lambda_2(A) = 2\cos(2\pi/n)$.

*Step 4.* The gap is

$$
\gamma = 2 - 2\cos\!\left( \frac{2\pi}{n} \right) = 4 \sin^{2}\!\left( \frac{\pi}{n} \right) = O(n^{-2}).
$$

$$
\boxed{\lambda^{(k)} = 2\cos(2\pi k/n), \qquad \lambda_2(A) = 2\cos(2\pi/n), \qquad \gamma = 4\sin^2(\pi/n)}
$$

**Key takeaway.** Two index conventions are in play and must not be mixed: $\lambda^{(k)}$ labels
a Fourier mode, $\lambda_2$ labels the second largest sorted eigenvalue. Because
$\gamma = O(n^{-2})$, a random walk on a cycle needs $\Theta(n^2)$ steps to mix — the signature
of diffusion.

In [27]:
for n in (6, 12, 40):
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i + 1) % n] = A[i, (i - 1) % n] = 1.0
    lam = np.linalg.eigvalsh(A)[::-1]
    modes = np.sort([2 * np.cos(2 * np.pi * k / n) for k in range(n)])[::-1]
    gap = 2 - lam[1]
    print(f"n={n:3d}  lambda_2 = {lam[1]:.6f}  2cos(2pi/n) = {2*np.cos(2*np.pi/n):.6f}"
          f"  gap = {gap:.6f}  4 sin^2(pi/n) = {4*np.sin(np.pi/n)**2:.6f}")
    assert np.allclose(lam, modes)
    assert abs(gap - 4 * np.sin(np.pi / n) ** 2) < 1e-10

n=  6  lambda_2 = 1.000000  2cos(2pi/n) = 1.000000  gap = 1.000000  4 sin^2(pi/n) = 1.000000
n= 12  lambda_2 = 1.732051  2cos(2pi/n) = 1.732051  gap = 0.267949  4 sin^2(pi/n) = 0.267949
n= 40  lambda_2 = 1.975377  2cos(2pi/n) = 1.975377  gap = 0.024623  4 sin^2(pi/n) = 0.024623


### Problem L2.8 — Normal modes of a two-mass spring chain (physics)

**Statement.** Two equal masses $m$ sit on a line between two walls, joined by three identical
springs of stiffness $k$ (wall, mass, mass, wall). Find the normal-mode frequencies and mode
shapes for small longitudinal displacements $x_1, x_2$.

**Intuition.** Normal modes are motions in which both masses oscillate at one frequency; they are
the eigenvectors of the stiffness matrix.

**Solution.**

*Step 1 — equations of motion.* Newton's second law gives

$$
m\ddot{x}_1 = -k x_1 + k(x_2 - x_1), \qquad m\ddot{x}_2 = -k(x_2 - x_1) - k x_2 ,
$$

that is $m\ddot{x} = -Kx$ with
$K = k \begin{pmatrix} 2 & -1 \\ -1 & 2 \end{pmatrix}$.

*Step 2 — substitute a normal mode.* Setting $x(t) = v e^{i\omega t}$ turns this into the
eigenvalue problem

$$
K v = m \omega^2 v .
$$

*Step 3 — diagonalize $K$.* $p_K(z) = (2k - z)^2 - k^2$, so the eigenvalues are $z = k$ and
$z = 3k$, with eigenvectors $(1,1)$ and $(1,-1)$ respectively.

*Step 4 — read off the frequencies.*

$$
\omega_1 = \sqrt{k/m} \ \text{(in phase)}, \qquad \omega_2 = \sqrt{3k/m} \ \text{(out of phase)} .
$$

$$
\boxed{\omega_1 = \sqrt{k/m}, \; v_1 = (1,1); \qquad \omega_2 = \sqrt{3k/m}, \; v_2 = (1,-1); \qquad \omega_2/\omega_1 = \sqrt3}
$$

**Key takeaway.** $K$ is symmetric because the spring forces derive from a potential, and $M$ is
positive definite because masses are positive; Theorem 4.2 is exactly what guarantees the modes
are real, orthogonal and non-interacting.

In [28]:
k_spring, m_mass = 2.5, 0.4
K = k_spring * np.array([[2.0, -1.0], [-1.0, 2.0]])
M = m_mass * np.eye(2)
Aop = np.linalg.inv(M) @ K
vals, modes = np.linalg.eigh(Aop)
omega = np.sqrt(vals)
print("omega          :", omega)
print("predicted      :", np.sqrt(k_spring / m_mass), np.sqrt(3 * k_spring / m_mass))
print("mode shapes    :\n", np.round(modes, 6))
print("frequency ratio:", omega[1] / omega[0], "  sqrt(3) =", np.sqrt(3))

# integrate the chain from the in-phase mode and measure the period directly
dt, steps = 1e-4, 60000
x, v = modes[:, 0].copy(), np.zeros(2)
crossings, prev = [], x.copy()
for step in range(steps):
    v = v - dt * (Aop @ x)
    x = x + dt * v
    if prev[0] < 0 <= x[0]:
        crossings.append(step * dt)
    prev = x.copy()
print("measured period:", crossings[1] - crossings[0],
      "   2 pi / omega_1 =", 2 * np.pi / omega[0])
assert abs((crossings[1] - crossings[0]) - 2 * np.pi / omega[0]) < 1e-3
assert abs(omega[0] - np.sqrt(k_spring / m_mass)) < 1e-12
assert abs(omega[1] - np.sqrt(3 * k_spring / m_mass)) < 1e-12
assert abs(omega[1] / omega[0] - np.sqrt(3)) < 1e-12

omega          : [2.5    4.3301]
predicted      : 2.5 4.330127018922194
mode shapes    :
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]
frequency ratio: 1.7320508075688774   sqrt(3) = 1.7320508075688772


measured period: 2.5133    2 pi / omega_1 = 2.5132741228718345


### Problem L2.9 — A two-level quantum system (physics)

**Statement.** A particle can sit in either of two symmetric wells, with Hamiltonian

$$
H = \begin{pmatrix} E_0 & -V \\ -V & E_0 \end{pmatrix}, \qquad V \gt 0 .
$$

Find the stationary energies and states, and the probability of finding the particle in well $2$
at time $t$ if it starts in well $1$.

**Intuition.** Tunnelling couples the two wells; the true stationary states are the symmetric and
antisymmetric combinations, and their energy splitting sets the tunnelling rate.

**Solution.**

*Step 1 — diagonalize.* $H$ is real symmetric, so Theorem 4.2 applies. Its eigenvalues are
$E_0 \mp V$ with eigenvectors

$$
\lvert s \rangle = \tfrac{1}{\sqrt2}(1,1)^{\top} \ (E_s = E_0 - V), \qquad
\lvert a \rangle = \tfrac{1}{\sqrt2}(1,-1)^{\top} \ (E_a = E_0 + V).
$$

*Step 2 — expand the initial state.*
$\lvert 1 \rangle = \tfrac{1}{\sqrt2}\bigl( \lvert s \rangle + \lvert a \rangle \bigr)$.

*Step 3 — evolve.* Each eigenstate picks up its own phase:

$$
\lvert \psi(t) \rangle = \tfrac{1}{\sqrt2}\left( e^{-iE_s t/\hbar} \lvert s \rangle + e^{-iE_a t/\hbar} \lvert a \rangle \right).
$$

*Step 4 — project onto well $2$.* Using
$\langle 2 \mid s \rangle = \tfrac{1}{\sqrt2}$ and
$\langle 2 \mid a \rangle = -\tfrac{1}{\sqrt2}$,

$$
\langle 2 \mid \psi(t) \rangle = \tfrac12 e^{-iE_0 t/\hbar} \left( e^{iVt/\hbar} - e^{-iVt/\hbar} \right) = i\, e^{-iE_0 t/\hbar} \sin(Vt/\hbar).
$$

$$
\boxed{E_{\pm} = E_0 \mp V, \qquad P_2(t) = \sin^2\!\left( \frac{Vt}{\hbar} \right), \qquad T = \frac{\pi \hbar}{V}}
$$

**Key takeaway.** The observable is self-adjoint, so Theorem 4.3 makes its eigenvalues real
(measurable energies) and its eigenstates orthogonal (mutually exclusive outcomes); the
oscillation period is inversely proportional to the energy splitting $2V$.

In [29]:
E0, Vc, hbar = 2.0, 0.5, 1.0
H = np.array([[E0, -Vc], [-Vc, E0]])
energies, states = np.linalg.eigh(H)
print("energies    :", energies, "   predicted", E0 - Vc, E0 + Vc)
print("eigenstates :\n", np.round(states, 6))

# time evolution through the spectral decomposition itself
psi0 = np.array([1.0 + 0j, 0.0 + 0j])
for t in [0.0, np.pi / (4 * Vc), np.pi / (2 * Vc), np.pi / Vc]:
    U = states @ np.diag(np.exp(-1j * energies * t / hbar)) @ states.T
    psi = U @ psi0
    print(f"  t = {t:7.4f}   P2 = {abs(psi[1])**2:.10f}   sin^2(Vt/hbar) = {np.sin(Vc*t/hbar)**2:.10f}")
    assert abs(abs(psi[1]) ** 2 - np.sin(Vc * t / hbar) ** 2) < 1e-12
print("period pi hbar / V =", np.pi * hbar / Vc)
assert np.allclose(np.sort(energies), [E0 - Vc, E0 + Vc])

energies    : [1.5 2.5]    predicted 1.5 2.5
eigenstates :
 [[-0.7071 -0.7071]
 [-0.7071  0.7071]]
  t =  0.0000   P2 = 0.0000000000   sin^2(Vt/hbar) = 0.0000000000
  t =  1.5708   P2 = 0.5000000000   sin^2(Vt/hbar) = 0.5000000000
  t =  3.1416   P2 = 1.0000000000   sin^2(Vt/hbar) = 1.0000000000
  t =  6.2832   P2 = 0.0000000000   sin^2(Vt/hbar) = 0.0000000000
period pi hbar / V = 6.283185307179586


### Problem L2.10 — The reduced operator of dynamic mode decomposition

**Statement.** Given snapshot matrices $X, X' \in \mathbb{R}^{n \times m}$, let
$X \approx U_r \Sigma_r V_r^{\top}$ be a rank-$r$ truncated SVD and define

$$
B = X' V_r \Sigma_r^{-1}, \qquad S = B U_r^{\top}, \qquad \tilde{S} = U_r^{\top} B .
$$

Prove that $\tilde{S}$ and $S$ have the same non-zero eigenvalues, and that if
$\tilde{S} w = \mu w$ with $\mu \neq 0$ then $\phi = Bw$ satisfies $S\phi = \mu \phi$.

**Intuition.** $S$ and $\tilde{S}$ are the two orders of the same product, and reversing a
product never changes the non-zero spectrum.

**Solution.**

*Step 1.* By construction $S = B U_r^{\top}$ and $\tilde{S} = U_r^{\top} B$ are the two products
of $B \in \mathbb{R}^{n \times r}$ and $U_r^{\top} \in \mathbb{R}^{r \times n}$ in the two orders.

*Step 2.* Problem L3.5 gives, for any $A \in \mathbb{R}^{p \times q}$ and
$B \in \mathbb{R}^{q \times p}$, the identity $z^{q} \det(zI_p - AB) = z^{p} \det(zI_q - BA)$.
Applied with $A = B$ of size $n \times r$ and $B = U_r^{\top}$ of size $r \times n$ it reads
$z^{r} \det(zI_n - S) = z^{n} \det(zI_r - \tilde{S})$, so $S$ and $\tilde{S}$ share every
non-zero eigenvalue with the same algebraic multiplicity, and $S$ carries $n - r$ extra zeros.

*Step 3 — the modes.* Let $\tilde{S} w = U_r^{\top} B w = \mu w$ with $\mu \neq 0$ and set
$\phi = Bw$. Then

$$
S \phi = B U_r^{\top} B w = B (\mu w) = \mu \phi .
$$

*Step 4.* $\phi \neq 0$, since $\phi = 0$ would give $\mu w = U_r^{\top} \phi = 0$ and hence
$w = 0$.

$$
\boxed{\operatorname{spec}(S) \setminus \{0\} = \operatorname{spec}(\tilde{S}) \setminus \{0\}, \qquad \phi = X' V_r \Sigma_r^{-1} w}
$$

**Key takeaway.** This is why DMD never forms the $n \times n$ operator: the $r \times r$ matrix
$\tilde{S}$ carries the entire non-trivial spectrum, and $\phi = Bw$ recovers the full-dimensional
mode in one matrix-vector product.

In [30]:
n_state, m_snap, r = 8, 12, 3
Ubasis, _ = np.linalg.qr(rng.standard_normal((n_state, r)))
Ctrue = rng.standard_normal((r, r))
z = rng.standard_normal((r, m_snap + 1))
for j in range(m_snap):
    z[:, j + 1] = Ctrue @ z[:, j]
data = Ubasis @ z
X, Xp = data[:, :m_snap], data[:, 1:]

U, sv, Vt = np.linalg.svd(X, full_matrices=False)
Ur, sr, Vr = U[:, :r], sv[:r], Vt[:r].T
B = Xp @ Vr @ np.diag(1.0 / sr)
S = B @ Ur.T
Stil = Ur.T @ B

ev_S = np.sort_complex(np.linalg.eigvals(S)[np.abs(np.linalg.eigvals(S)) > 1e-8])
ev_T = np.sort_complex(np.linalg.eigvals(Stil))
print("non-zero eig(S) :", np.round(ev_S, 8))
print("eig(S tilde)    :", np.round(ev_T, 8))
print("true system eig :", np.round(np.sort_complex(np.linalg.eigvals(Ctrue)), 8))
mu, w = np.linalg.eig(Stil)
phi = B @ w
res = [np.linalg.norm(S @ phi[:, i] - mu[i] * phi[:, i]) for i in range(r)]
print("mode residuals  :", np.round(res, 15))
assert np.allclose(ev_S, ev_T)
assert max(res) < 1e-10

non-zero eig(S) : [-0.4353-0.5201j -0.4353+0.5201j  1.5125+0.j    ]
eig(S tilde)    : [-0.4353-0.5201j -0.4353+0.5201j  1.5125+0.j    ]
true system eig : [-0.4353-0.5201j -0.4353+0.5201j  1.5125+0.j    ]
mode residuals  : [0. 0. 0.]


## L3 — Challenge Proofs

### Problem L3.1 — Rayleigh characterization of the extreme eigenvalues

**Statement.** For real symmetric $A$ with $\lambda_1 \ge \dots \ge \lambda_n$, prove

$$
\lambda_1 = \max_{x \neq 0} \frac{x^{\top} A x}{x^{\top} x}, \qquad
\lambda_n = \min_{x \neq 0} \frac{x^{\top} A x}{x^{\top} x} .
$$

**Intuition.** In the eigenbasis the Rayleigh quotient is a weighted average of the eigenvalues,
and an average never leaves the range of the values.

**Solution.**

*Step 1.* By Theorem 4.2 take an orthonormal eigenbasis $q_1, \dots, q_n$ and write
$x = \sum_i c_i q_i$ with not all $c_i$ zero.

*Step 2.* Orthonormality gives $x^{\top} x = \sum_i c_i^2$ and
$x^{\top} A x = \sum_i \lambda_i c_i^2$, so

$$
R_A(x) = \frac{\sum_i \lambda_i c_i^2}{\sum_i c_i^2},
$$

a convex combination of the eigenvalues with weights $c_i^2 / \sum_j c_j^2$.

*Step 3.* Therefore $\lambda_n \le R_A(x) \le \lambda_1$ for every $x \neq 0$.

*Step 4.* Both bounds are attained: $R_A(q_1) = \lambda_1$ and $R_A(q_n) = \lambda_n$.

$$
\boxed{\lambda_1 = \max_{x \neq 0} R_A(x), \qquad \lambda_n = \min_{x \neq 0} R_A(x)}
$$

**Key takeaway.** This is the $k=1$ and $k=n$ case of Theorem 4.4, and it is what turns an
eigenvalue problem into an optimization problem — the starting point of every variational
eigensolver.

In [31]:
Mr = rng.standard_normal((5, 5))
A = (Mr + Mr.T) / 2
lam = np.linalg.eigvalsh(A)
best, worst = -np.inf, np.inf
for _ in range(20000):
    x = rng.standard_normal(5)
    r = x @ A @ x / (x @ x)
    best, worst = max(best, r), min(worst, r)
print("lambda_1, lambda_n :", lam[-1], lam[0])
print("sampled max, min   :", best, worst)
print("sampling stays inside the interval:", worst >= lam[0] - 1e-12 and best <= lam[-1] + 1e-12)
assert worst >= lam[0] - 1e-12 and best <= lam[-1] + 1e-12

lambda_1, lambda_n : 2.2688426482486266 -1.578011729268127
sampled max, min   : 2.239764874883536 -1.571376431565328
sampling stays inside the interval: True


### Problem L3.2 — Commuting diagonalizable matrices share an eigenbasis

**Statement.** Let $A, B \in \mathbb{C}^{n \times n}$ both be diagonalizable. Prove that
$AB = BA$ if and only if they are simultaneously diagonalizable.

**Intuition.** Commuting means $B$ preserves the eigenspaces of $A$, so $B$ can be diagonalized
inside each of them separately.

**Solution.**

*Step 1 — the easy direction.* If $A = PD_AP^{-1}$ and $B = PD_BP^{-1}$ then
$AB = PD_AD_BP^{-1} = PD_BD_AP^{-1} = BA$, since diagonal matrices commute.

*Step 2 — eigenspaces are invariant.* Assume $AB = BA$, let $E_\lambda$ be an eigenspace of $A$
and $v \in E_\lambda$. Then

$$
A(Bv) = B(Av) = \lambda (Bv),
$$

so $Bv \in E_\lambda$ and $E_\lambda$ is $B$-invariant.

*Step 3 — a lemma on restrictions.* A matrix is diagonalizable if and only if it is annihilated
by a product of **distinct** linear factors. One direction: if $B$ is diagonalizable with distinct
eigenvalues $\mu_1, \dots, \mu_r$, then $q(z) = \prod_{i}(z - \mu_i)$ kills every eigenvector and
hence $q(B) = 0$.

The converse: suppose $q(B) = 0$ with the $\mu_i$ distinct, and let

$$
e_i(z) = \prod_{j \neq i} \frac{z - \mu_j}{\mu_i - \mu_j} .
$$

Both $\sum_i e_i(z)$ and the constant $1$ are polynomials of degree at most $r-1$ agreeing at the
$r$ points $\mu_1, \dots, \mu_r$, so $\sum_i e_i = 1$ identically and $\sum_i e_i(B) = I$. Also
$(B - \mu_i I) e_i(B) = q(B) / \prod_{j \neq i}(\mu_i - \mu_j) = 0$, so
$e_i(B)x \in \operatorname{Null}(B - \mu_i I)$. Hence every $x = \sum_i e_i(B) x$ is a sum of
eigenvectors and $B$ is diagonalizable.

*Step 4 — restrict.* $B$ diagonalizable gives $q(B) = 0$ with distinct linear factors; the same
$q$ annihilates $B|_{E_\lambda}$, so by Step 3 the restriction is diagonalizable and $E_\lambda$
has a basis of $B$-eigenvectors.

*Step 5.* $A$ is diagonalizable, so $\mathbb{C}^n$ is the direct sum of its eigenspaces.
Concatenating the bases from Step 4 gives a basis of common eigenvectors.

$$
\boxed{AB = BA \iff A \text{ and } B \text{ are simultaneously diagonalizable}}
$$

**Key takeaway.** Step 3 is the part the usual textbook sketch skips; without it "the restriction
is also diagonalizable" is an assertion. In quantum mechanics this theorem is the statement that
commuting observables are simultaneously measurable.

In [32]:
Qc, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A = Qc @ np.diag([3.0, 3.0, -1.0, 2.0]) @ Qc.T
B = Qc @ np.diag([1.0, 5.0, 5.0, 0.5]) @ Qc.T
print("||AB - BA||           :", np.linalg.norm(A @ B - B @ A))
w, V = np.linalg.eig(A + 1e-3 * B)      # a generic combination separates the eigenvalues
print("common eigenvector residuals:")
for i in range(4):
    v = V[:, i].real
    ra = np.linalg.norm(A @ v - (v @ A @ v) * v)
    rb = np.linalg.norm(B @ v - (v @ B @ v) * v)
    print(f"  vector {i}: ||Av - a v|| = {ra:.2e}   ||Bv - b v|| = {rb:.2e}")
    assert ra < 1e-9 and rb < 1e-9
Cn = rng.standard_normal((4, 4))
print("\nnon-commuting pair ||AC - CA||:", np.linalg.norm(A @ Cn - Cn @ A))
assert np.linalg.norm(A @ B - B @ A) < 1e-12

||AB - BA||           : 4.545817630668578e-15
common eigenvector residuals:
  vector 0: ||Av - a v|| = 5.33e-16   ||Bv - b v|| = 1.09e-15
  vector 1: ||Av - a v|| = 1.24e-15   ||Bv - b v|| = 2.34e-15
  vector 2: ||Av - a v|| = 1.96e-15   ||Bv - b v|| = 6.94e-14
  vector 3: ||Av - a v|| = 1.25e-15   ||Bv - b v|| = 4.22e-13

non-commuting pair ||AC - CA||: 9.073756890067438


### Problem L3.3 — A symmetric cube root of the identity

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ be symmetric with $A^3 = I$. Prove $A = I$.

**Intuition.** Symmetry forces the eigenvalues to be real, and $1$ is the only real cube root of
$1$.

**Solution.**

*Step 1.* By Theorem 4.2, $A = Q\Lambda Q^{\top}$ with $Q$ orthogonal and $\Lambda$ real
diagonal.

*Step 2.* $A^3 = Q\Lambda^3 Q^{\top} = I$ forces $\Lambda^3 = I$, that is $\lambda_i^3 = 1$ for
every $i$.

*Step 3.* The equation $\lambda^3 = 1$ has the three complex roots
$1, e^{2\pi i/3}, e^{-2\pi i/3}$, of which only $\lambda = 1$ is real. Since Theorem 4.2 makes
every $\lambda_i$ real, $\Lambda = I$.

*Step 4.* Hence $A = Q I Q^{\top} = QQ^{\top} = I$.

$$
\boxed{A = I}
$$

**Key takeaway.** Symmetry is essential: the rotation by $120$ degrees satisfies $R^3 = I$ with
$R \neq I$, because its eigenvalues are the two non-real cube roots of unity.

In [33]:
Qs, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A = Qs @ np.diag([1.0, 1.0, 1.0, 1.0]) @ Qs.T
print("symmetric case: ||A^3 - I|| =", np.linalg.norm(np.linalg.matrix_power(A, 3) - np.eye(4)),
      "  ||A - I|| =", np.linalg.norm(A - np.eye(4)))
th = 2 * np.pi / 3
R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
print("rotation by 120 deg: ||R^3 - I|| =", np.linalg.norm(np.linalg.matrix_power(R, 3) - np.eye(2)),
      "  ||R - I|| =", np.linalg.norm(R - np.eye(2)))
print("R eigenvalues:", np.round(np.linalg.eigvals(R), 8), " (non-real, so R is not symmetric)")
assert np.linalg.norm(A - np.eye(4)) < 1e-12
assert np.linalg.norm(np.linalg.matrix_power(R, 3) - np.eye(2)) < 1e-12
assert np.linalg.norm(R - np.eye(2)) > 1.0

symmetric case: ||A^3 - I|| = 7.927967630621931e-16   ||A - I|| = 2.642655876873977e-16
rotation by 120 deg: ||R^3 - I|| = 9.235218158431266e-16   ||R - I|| = 2.449489742783178
R eigenvalues: [-0.5+0.866j -0.5-0.866j]  (non-real, so R is not symmetric)


### Problem L3.4 — Vanishing trace powers force nilpotency

**Statement.** (a) Let $A \in \mathbb{C}^{n \times n}$ satisfy $\operatorname{tr}(A^k) = 0$ for
$k = 1, \dots, n$. Prove $A$ is nilpotent. (b) Deduce that $AB - BA = A$ forces $A$ nilpotent.

**Intuition.** The power sums of the eigenvalues determine the characteristic polynomial; if they
all vanish, the polynomial is $z^n$.

**Solution.**

*Step 1 — set up.* Let $\lambda_1, \dots, \lambda_n$ be the eigenvalues of $A$ with algebraic
multiplicity. By Problem L1.2 applied to $A^k$, whose eigenvalues are $\lambda_i^k$, the power
sums are $p_k = \sum_i \lambda_i^k = \operatorname{tr}(A^k)$.

*Step 2 — Newton's identities.* For $1 \le k \le n$,

$$
k \, e_k = \sum_{i=1}^{k} (-1)^{i-1} e_{k-i} \, p_i ,
$$

where $e_j$ is the $j$-th elementary symmetric polynomial in the $\lambda_i$ and $e_0 = 1$.
Working over $\mathbb{C}$ makes the division by $k$ legitimate.

*Step 3 — induct.* $p_1 = \dots = p_n = 0$ makes the right-hand side zero for every
$k \le n$, so $e_1 = e_2 = \dots = e_n = 0$.

*Step 4 — conclude (a).* The characteristic polynomial is
$\prod_i (z - \lambda_i) = z^n - e_1 z^{n-1} + \dots + (-1)^n e_n = z^n$, so every eigenvalue is
$0$, and Cayley-Hamilton gives $A^n = 0$.

*Step 5 — the commutator identity for (b).* From $AB - BA = A$ prove
$A^k B - B A^k = k A^k$ by induction. The case $k=1$ is the hypothesis. Assuming it for $k$ and
multiplying on the left by $A$,

$$
A^{k+1} B - ABA^k = kA^{k+1}.
$$

Substituting $AB = BA + A$ gives $ABA^k = BA^{k+1} + A^{k+1}$, hence
$A^{k+1}B - BA^{k+1} = (k+1)A^{k+1}$.

*Step 6 — take traces.* $\operatorname{tr}(A^kB) = \operatorname{tr}(BA^k)$, so
$k \operatorname{tr}(A^k) = 0$ and therefore $\operatorname{tr}(A^k) = 0$ for every $k \ge 1$.
Part (a) applies.

$$
\boxed{\operatorname{tr}(A^k) = 0 \ \forall k \le n \implies A^n = 0; \qquad AB - BA = A \implies A \text{ nilpotent}}
$$

**Key takeaway.** Traces of powers are coordinate-free, so they see the spectrum without ever
solving for it — the same idea underlies stochastic trace estimators for log-determinants in
machine learning.

In [34]:
N = np.array([[0.0, 2.0, 5.0], [0.0, 0.0, 3.0], [0.0, 0.0, 0.0]])
print("traces of powers:", [float(np.trace(np.linalg.matrix_power(N, k))) for k in (1, 2, 3)])
print("N^3             :\n", np.linalg.matrix_power(N, 3))

# a concrete solution of AB - BA = A: the shift and the grading operator
A = np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 0.0, 0.0]])
B = np.diag([0.0, 1.0, 2.0])
print("\n||AB - BA - A|| :", np.linalg.norm(A @ B - B @ A - A))
print("traces of powers:", [float(np.trace(np.linalg.matrix_power(A, k))) for k in (1, 2, 3)])
print("A^3             :\n", np.linalg.matrix_power(A, 3))
assert np.allclose(np.linalg.matrix_power(N, 3), 0.0)
assert np.allclose(A @ B - B @ A, A)
assert np.allclose(np.linalg.matrix_power(A, 3), 0.0)

traces of powers: [0.0, 0.0, 0.0]
N^3             :
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

||AB - BA - A|| : 0.0
traces of powers: [0.0, 0.0, 0.0]
A^3             :
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


### Problem L3.5 — $AB$ and $BA$ share their non-zero spectrum

**Statement.** For $A \in \mathbb{C}^{p \times q}$ and $B \in \mathbb{C}^{q \times p}$, prove

$$
z^{q} \det(z I_p - AB) = z^{p} \det(z I_q - BA),
$$

so $AB$ and $BA$ have the same non-zero eigenvalues with the same algebraic multiplicities.

**Intuition.** The two block factorizations of one block matrix, in the two possible orders,
expose the two characteristic polynomials.

**Solution.**

*Step 1.* Consider

$$
M = \begin{pmatrix} z I_p & A \\ B & I_q \end{pmatrix}, \qquad
L = \begin{pmatrix} I_p & 0 \\ -B & z I_q \end{pmatrix}.
$$

*Step 2.* Multiplying in one order,

$$
ML = \begin{pmatrix} z I_p - AB & zA \\ 0 & z I_q \end{pmatrix},
\qquad \det(ML) = \det(z I_p - AB) \cdot z^{q}.
$$

*Step 3.* Multiplying in the other order,

$$
LM = \begin{pmatrix} z I_p & A \\ 0 & z I_q - BA \end{pmatrix},
\qquad \det(LM) = z^{p} \cdot \det(z I_q - BA).
$$

*Step 4.* $\det(ML) = \det M \det L = \det(LM)$, which gives the identity.

*Step 5.* For $z \neq 0$ the powers $z^p, z^q$ are non-zero, so
$\det(zI_p - AB) = 0 \iff \det(zI_q - BA) = 0$, with matching multiplicities.

$$
\boxed{\operatorname{spec}(AB) \setminus \{0\} = \operatorname{spec}(BA) \setminus \{0\}}
$$

**Key takeaway.** This is what lets an algorithm work with the smaller of $A^{\top}A$ and
$AA^{\top}$ — the reason PCA on $m \ll n$ samples costs $O(m^3)$, not $O(n^3)$, and the reason
DMD in Problem L2.10 never forms its big operator.

In [35]:
p_dim, q_dim = 5, 3
A = rng.standard_normal((p_dim, q_dim))
B = rng.standard_normal((q_dim, p_dim))
ev_ab = np.sort_complex(np.linalg.eigvals(A @ B))
ev_ba = np.sort_complex(np.linalg.eigvals(B @ A))
print("eig(AB) (5x5):", np.round(ev_ab, 8))
print("eig(BA) (3x3):", np.round(ev_ba, 8))
nz = np.sort_complex(ev_ab[np.abs(ev_ab) > 1e-8])
print("non-zero part of eig(AB):", np.round(nz, 8))
z = 1.7
lhs = z ** q_dim * np.linalg.det(z * np.eye(p_dim) - A @ B)
rhs = z ** p_dim * np.linalg.det(z * np.eye(q_dim) - B @ A)
print(f"identity at z = {z}: {lhs:.10f} vs {rhs:.10f}")
assert np.allclose(nz, ev_ba)
assert abs(lhs - rhs) < 1e-8

eig(AB) (5x5): [-2.7569+0.j     -0.    +0.j      0.    +0.j      0.7109-0.9818j
  0.7109+0.9818j]
eig(BA) (3x3): [-2.7569+0.j      0.7109-0.9818j  0.7109+0.9818j]
non-zero part of eig(AB): [-2.7569+0.j      0.7109-0.9818j  0.7109+0.9818j]
identity at z = 1.7: 122.9078638625 vs 122.9078638625


### Problem L3.6 — Skew-symmetric matrices

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ satisfy $A^{\top} = -A$. Prove every
eigenvalue is purely imaginary, and that $\det A = 0$ when $n$ is odd.

**Intuition.** Skew-symmetry makes the quadratic form $v^{\ast} A v$ purely imaginary, which
pins the eigenvalues to the imaginary axis.

**Solution.**

*Step 1.* Let $Av = \lambda v$ with $v \in \mathbb{C}^n$, $v \neq 0$. Then
$v^{\ast} A v = \lambda \lVert v \rVert^2$.

*Step 2.* Since $A$ is real, $A^{\ast} = A^{\top} = -A$, so

$$
\overline{v^{\ast} A v} = (v^{\ast} A v)^{\ast} = v^{\ast} A^{\ast} v = - v^{\ast} A v .
$$

A number equal to minus its own conjugate is purely imaginary.

*Step 3.* Therefore $\lambda = v^{\ast} A v / \lVert v \rVert^2$ is purely imaginary,
$\lambda = i\omega$ with $\omega \in \mathbb{R}$.

*Step 4 — odd $n$.* Using $\det(A^{\top}) = \det(A)$ and $\det(cA) = c^n \det A$,

$$
\det A = \det(A^{\top}) = \det(-A) = (-1)^n \det A = -\det A
$$

for odd $n$, so $\det A = 0$.

$$
\boxed{\operatorname{Re}\lambda = 0 \text{ for every eigenvalue}; \qquad n \text{ odd} \implies \det A = 0}
$$

**Key takeaway.** Skew-symmetric matrices are normal, so Theorem 4.3 diagonalizes them
unitarily — over $\mathbb{C}$, not over $\mathbb{R}$. In odd dimensions the forced zero
eigenvalue is the rotation axis of a rigid-body angular velocity.

In [36]:
for n in (3, 4, 5):
    Mr = rng.standard_normal((n, n))
    A = (Mr - Mr.T) / 2
    ev = np.linalg.eigvals(A)
    print(f"n={n}  max |Re(lambda)| = {np.abs(ev.real).max():.3e}   det = {np.linalg.det(A):.3e}"
          f"   normal: {np.allclose(A.T @ A, A @ A.T)}")
    assert np.abs(ev.real).max() < 1e-12
    if n % 2 == 1:
        assert abs(np.linalg.det(A)) < 1e-10

n=3  max |Re(lambda)| = 1.584e-17   det = 0.000e+00   normal: True
n=4  max |Re(lambda)| = 1.388e-17   det = 3.307e-03   normal: True
n=5  max |Re(lambda)| = 1.110e-16   det = -1.189e-16   normal: True


### Problem L3.7 — Orthogonal matrices have unimodular spectrum

**Statement.** Let $Q \in \mathbb{R}^{n \times n}$ satisfy $Q^{\top}Q = I$. Prove
$\lvert \lambda \rvert = 1$ for every eigenvalue $\lambda \in \mathbb{C}$ of $Q$.

**Intuition.** An orthogonal matrix preserves lengths, so it cannot scale any direction, real or
complex.

**Solution.**

*Step 1.* Let $Qv = \lambda v$ with $v \in \mathbb{C}^n$, $v \neq 0$.

*Step 2.* Since $Q$ is real, $Q^{\ast} = Q^{\top}$, so

$$
(Qv)^{\ast}(Qv) = v^{\ast} Q^{\top} Q v = v^{\ast} v = \lVert v \rVert^2 .
$$

*Step 3.* On the other hand $(Qv)^{\ast}(Qv) = (\lambda v)^{\ast}(\lambda v) = \lvert \lambda \rvert^2 \lVert v \rVert^2$.

*Step 4.* Dividing by $\lVert v \rVert^2 \gt 0$ gives $\lvert \lambda \rvert^2 = 1$.

$$
\boxed{\lvert \lambda \rvert = 1}
$$

**Key takeaway.** The spectrum sits on the unit circle, so $\rho(Q) = 1$ and $\lVert Q^k \rVert$
never grows — the numerical-stability argument for using orthogonal transformations in QR,
Householder and Givens algorithms.

In [37]:
Qo, _ = np.linalg.qr(rng.standard_normal((5, 5)))
ev = np.linalg.eigvals(Qo)
print("Q^T Q - I  :", np.linalg.norm(Qo.T @ Qo - np.eye(5)))
print("eigenvalues:", np.round(ev, 6))
print("moduli     :", np.round(np.abs(ev), 12))
print("||Q^100||_2:", np.linalg.norm(np.linalg.matrix_power(Qo, 100), 2))
assert np.allclose(np.abs(ev), 1.0)
assert abs(np.linalg.norm(np.linalg.matrix_power(Qo, 100), 2) - 1.0) < 1e-8

Q^T Q - I  : 6.63024967935515e-16
eigenvalues: [ 1.    +0.j     -0.0493+0.9988j -0.0493-0.9988j -0.9324+0.3614j
 -0.9324-0.3614j]
moduli     : [1. 1. 1. 1. 1.]
||Q^100||_2: 1.0000000000000009


### Problem L3.8 — Real solutions of $A^2 + I = 0$

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ satisfy $A^2 = -I$. Prove $n$ is even and
$\operatorname{tr} A = 0$.

**Intuition.** Such an $A$ is a real matrix representation of the imaginary unit, and $i$ needs
two real dimensions.

**Solution.**

*Step 1.* $A$ is annihilated by $z^2 + 1 = (z - i)(z + i)$, a product of distinct linear factors,
so by the lemma in Problem L3.2 $A$ is diagonalizable over $\mathbb{C}$ with spectrum contained
in $\{i, -i\}$.

*Step 2.* $A$ is real, so its characteristic polynomial has real coefficients and non-real roots
occur in conjugate pairs with equal algebraic multiplicity. Let $k$ be the common multiplicity of
$i$ and $-i$.

*Step 3.* Then $n = k + k = 2k$ is even.

*Step 4.* By Problem L1.2, $\operatorname{tr} A = k \cdot i + k \cdot (-i) = 0$.

$$
\boxed{n = 2k \text{ is even}, \qquad \operatorname{tr} A = 0}
$$

**Key takeaway.** The canonical example is the block-diagonal matrix of $90$-degree rotations;
this is exactly the statement that $\mathbb{C}$ acting on $\mathbb{R}^n$ makes $\mathbb{R}^n$ a
complex vector space of dimension $n/2$.

In [38]:
J2 = np.array([[0.0, -1.0], [1.0, 0.0]])
for k in (1, 2, 3):
    A = np.kron(np.eye(k), J2)
    print(f"n = {2*k}:  ||A^2 + I|| = {np.linalg.norm(A @ A + np.eye(2*k)):.2e}"
          f"   trace = {np.trace(A):.1f}   eigenvalues = {np.round(np.linalg.eigvals(A), 6)}")
    assert np.allclose(A @ A, -np.eye(2 * k))
    assert abs(np.trace(A)) < 1e-12

n = 2:  ||A^2 + I|| = 0.00e+00   trace = 0.0   eigenvalues = [0.+1.j 0.-1.j]
n = 4:  ||A^2 + I|| = 0.00e+00   trace = 0.0   eigenvalues = [0.+1.j 0.-1.j 0.+1.j 0.-1.j]
n = 6:  ||A^2 + I|| = 0.00e+00   trace = 0.0   eigenvalues = [0.+1.j 0.-1.j 0.+1.j 0.-1.j 0.+1.j 0.-1.j]


### Problem L3.9 — The product of two positive definite matrices

**Statement.** Let $A, B \in \mathbb{R}^{n \times n}$ be symmetric positive definite. Prove every
eigenvalue of $AB$ is real and strictly positive, even though $AB$ is generally not symmetric.

**Intuition.** $AB$ is similar to the symmetric matrix $A^{1/2} B A^{1/2}$, and similarity
preserves the spectrum.

**Solution.**

*Step 1 — the square root.* By Theorem 4.2, $A = Q\Lambda Q^{\top}$ with $\lambda_i \gt 0$.
Define $A^{1/2} = Q \Lambda^{1/2} Q^{\top}$, which is symmetric positive definite and satisfies
$(A^{1/2})^2 = A$.

*Step 2 — the similarity.* Since $A^{1/2}$ is invertible,

$$
A^{-1/2} (AB) A^{1/2} = A^{1/2} B A^{1/2} =: M .
$$

*Step 3 — $M$ is symmetric.* $M^{\top} = (A^{1/2})^{\top} B^{\top} (A^{1/2})^{\top} = A^{1/2} B A^{1/2} = M$.

*Step 4 — $M$ is positive definite.* For $x \neq 0$, $A^{1/2}x \neq 0$, so
$x^{\top} M x = (A^{1/2}x)^{\top} B (A^{1/2}x) \gt 0$.

*Step 5.* By Theorem 4.2 the spectrum of $M$ is real and positive, and similar matrices have the
same spectrum.

$$
\boxed{\operatorname{spec}(AB) \subset (0, \infty)}
$$

**Key takeaway.** Preconditioned systems $M^{-1}A x = M^{-1}b$ have exactly this structure, which
is why conjugate gradients still applies even though $M^{-1}A$ is not symmetric.

In [39]:
def rand_spd(n):
    Z = rng.standard_normal((n, n))
    return Z @ Z.T + n * np.eye(n)

A, B = rand_spd(4), rand_spd(4)
ev = np.linalg.eigvals(A @ B)
print("AB symmetric      :", np.allclose(A @ B, (A @ B).T))
print("eig(AB)           :", np.round(np.sort_complex(ev), 8))
print("max |Im|          :", np.abs(ev.imag).max())
print("min Re            :", ev.real.min())
lam_A, Q_A = np.linalg.eigh(A)
Ahalf = Q_A @ np.diag(np.sqrt(lam_A)) @ Q_A.T
M = Ahalf @ B @ Ahalf
print("eig(A^1/2 B A^1/2):", np.round(np.linalg.eigvalsh(M), 8))
assert np.abs(ev.imag).max() < 1e-9
assert ev.real.min() > 0
assert np.allclose(np.sort(ev.real), np.sort(np.linalg.eigvalsh(M)))

AB symmetric      : False
eig(AB)           : [ 21.948 +0.j  23.5655+0.j  72.68  +0.j 170.2743+0.j]
max |Im|          : 0.0
min Re            : 21.94799572228664
eig(A^1/2 B A^1/2): [ 21.948   23.5655  72.68   170.2743]


### Problem L3.10 — The matrix of minima is positive definite

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ have entries $a_{ij} = \min(i,j)$. Prove every
eigenvalue of $A$ is strictly positive.

**Intuition.** $A$ is the covariance of Brownian motion sampled at integer times, and its inverse
is the discrete second-difference operator.

**Solution.**

*Step 1 — guess the inverse.* Let $T$ be tridiagonal with $T_{ii} = 2$ for $i \lt n$,
$T_{nn} = 1$, and $T_{i,i+1} = T_{i+1,i} = -1$. A direct multiplication gives $AT = I$, so
$A^{-1} = T$.

*Step 2 — the quadratic form of $T$.* For any $x \in \mathbb{R}^n$,

$$
x^{\top} T x = x_1^2 + \sum_{i=1}^{n-1} (x_i - x_{i+1})^2 .
$$

Expanding the right-hand side reproduces $2\sum_{i \lt n} x_i^2 + x_n^2 - 2\sum_{i \lt n} x_i x_{i+1}$,
which is exactly $x^{\top} T x$.

*Step 3 — positivity.* The sum of squares is zero only if $x_1 = 0$ and $x_i = x_{i+1}$ for every
$i$, forcing $x = 0$. So $T \succ 0$.

*Step 4 — invert.* $A = T^{-1}$ is symmetric, and by Problem L0.4 its eigenvalues are the
reciprocals of those of $T$, hence all positive.

*Step 5 — the closed form.* Diagonalizing $T$ by discrete sine modes gives

$$
\lambda_k(A) = \frac{1}{4 \sin^2 \left( \dfrac{(2k-1)\pi}{2(2n+1)} \right)}, \qquad k = 1, \dots, n .
$$

$$
\boxed{A \succ 0, \qquad \lambda_k(A) = \tfrac{1}{4} \csc^2 \!\left( \tfrac{(2k-1)\pi}{2(2n+1)} \right)}
$$

**Key takeaway.** Positive definiteness is often easiest to prove on the *inverse*: here the
inverse is sparse and its quadratic form is visibly a sum of squares.

In [40]:
for n in (5, 7, 10):
    A = np.fromfunction(lambda i, j: np.minimum(i, j) + 1, (n, n), dtype=int).astype(float)
    T = np.diag(2.0 * np.ones(n)) + np.diag(-np.ones(n - 1), 1) + np.diag(-np.ones(n - 1), -1)
    T[n - 1, n - 1] = 1.0
    ev = np.sort(np.linalg.eigvalsh(A))[::-1]
    closed = np.array([1 / (4 * np.sin((2 * k - 1) * np.pi / (2 * (2 * n + 1))) ** 2)
                       for k in range(1, n + 1)])
    print(f"n={n:2d}  ||A T - I|| = {np.linalg.norm(A @ T - np.eye(n)):.2e}"
          f"   min eig = {ev.min():.6f}   max |eig - closed form| = {np.abs(ev - closed).max():.2e}")
    assert np.linalg.norm(A @ T - np.eye(n)) < 1e-10
    assert ev.min() > 0
    assert np.abs(ev - closed).max() < 1e-10

n= 5  ||A T - I|| = 0.00e+00   min eig = 0.271554   max |eig - closed form| = 2.22e-16
n= 7  ||A T - I|| = 0.00e+00   min eig = 0.261295   max |eig - closed form| = 5.33e-15
n=10  ||A T - I|| = 0.00e+00   min eig = 0.255680   max |eig - closed form| = 2.44e-15


### Problem L3.11 — The first two even moments of Wigner's semicircle law

**Statement.** Let $H_n$ be a GOE matrix: symmetric, with $H_{ij} \sim N(0,1)$ independent for
$i \lt j$ and $H_{ii} \sim N(0,2)$. Put $W_n = H_n / \sqrt{n}$. Show

$$
\frac{1}{n} \mathbb{E} \operatorname{tr}(W_n^{2}) = 1 + \frac{1}{n} \longrightarrow C_1 = 1,
\qquad
\frac{1}{n} \mathbb{E} \operatorname{tr}(W_n^{4}) \longrightarrow C_2 = 2 ,
$$

matching the second and fourth moments of the semicircle density
$\rho(x) = \tfrac{1}{2\pi}\sqrt{4 - x^2}$ on $[-2,2]$.

**Intuition.** $\tfrac1n \operatorname{tr}(W^{2k})$ is the average of $\lambda^{2k}$ over the
spectrum, so its limit is the $2k$-th moment of the limiting spectral density. Gaussian
expectations reduce to counting index patterns.

**Solution.**

*Step 1 — the second moment.*

$$
\frac{1}{n}\mathbb{E}\operatorname{tr}(W_n^2) = \frac{1}{n^2} \mathbb{E} \sum_{i,j} H_{ij}^2
= \frac{1}{n^2}\Bigl[ n(n-1) \cdot 1 + n \cdot 2 \Bigr] = \frac{n+1}{n},
$$

using variance $1$ off the diagonal and $2$ on it. This tends to $1 = C_1$.

*Step 2 — set up the fourth moment.*

$$
\mathbb{E}\operatorname{tr}(H_n^4) = \sum_{i,j,k,l} \mathbb{E}\bigl[ H_{ij} H_{jk} H_{kl} H_{li} \bigr].
$$

*Step 3 — only pairings survive.* The entries are centred and independent up to symmetry, so a
term vanishes unless the four factors split into two matching pairs of unordered indices.

*Step 4 — count the leading families.* There are exactly two families with three free indices,
hence $n^3$ terms each:

- $\{i,j\} = \{j,k\}$ and $\{k,l\} = \{l,i\}$, which forces $i = k$ and leaves $i, j, l$ free;
- $\{i,j\} = \{l,i\}$ and $\{j,k\} = \{k,l\}$, which forces $j = l$ and leaves $i, j, k$ free.

Each such term contributes $\mathbb{E}[H_{ij}^2]\,\mathbb{E}[H_{il}^2] = 1$ when the indices are
distinct.

*Step 5 — everything else is smaller.* The remaining pairing needs $i = l$ and $j = k$, giving
$\mathbb{E}[H_{ij}^4] = 3$ but only $n^2$ terms; the overlap of the two families above (where
$i = k$ **and** $j = l$) is also $O(n^2)$; and diagonal coincidences contribute $O(n^2)$.

*Step 6 — conclude.*

$$
\frac{1}{n}\mathbb{E}\operatorname{tr}(W_n^4) = \frac{1}{n^3}\mathbb{E}\operatorname{tr}(H_n^4) = \frac{2n^3 + O(n^2)}{n^3} \longrightarrow 2 = C_2 .
$$

$$
\boxed{\tfrac1n \mathbb{E}\operatorname{tr}(W_n^{2}) = 1 + \tfrac1n \to 1, \qquad \tfrac1n \mathbb{E}\operatorname{tr}(W_n^{4}) \to 2}
$$

**Key takeaway.** The surviving index patterns are exactly the non-crossing pairings of $2k$
points, which is why the limits are the Catalan numbers $C_k$ and why the semicircle law plays
the role the normal law plays for scalars.

In [41]:
# semicircle moments by quadrature
from numpy.polynomial.legendre import leggauss
nodes, weights = leggauss(400)
nodes, weights = 2 * nodes, 2 * weights
dens = np.sqrt(np.maximum(0.0, 4 - nodes ** 2)) / (2 * np.pi)
print("semicircle moments: m2 =", (weights * dens * nodes ** 2).sum(),
      "  m4 =", (weights * dens * nodes ** 4).sum())

for n, reps in [(100, 200), (400, 40), (1000, 8)]:
    m2, m4 = [], []
    for _ in range(reps):
        Z = rng.standard_normal((n, n))
        H = (Z + Z.T) / np.sqrt(2)          # off-diagonal variance 1, diagonal variance 2
        ev = np.linalg.eigvalsh(H / np.sqrt(n))
        m2.append((ev ** 2).mean())
        m4.append((ev ** 4).mean())
    print(f"n={n:5d} reps={reps:4d}   m2 = {np.mean(m2):.5f} (exact {1 + 1/n:.5f})"
          f"   m4 = {np.mean(m4):.5f} (limit 2)")
assert abs(np.mean(m2) - 1.0) < 0.02
assert abs(np.mean(m4) - 2.0) < 0.05

semicircle moments: m2 = 1.0000000326500724   m4 = 2.0000001306028423
n=  100 reps= 200   m2 = 1.00953 (exact 1.01000)   m4 = 2.05072 (limit 2)


n=  400 reps=  40   m2 = 1.00157 (exact 1.00250)   m4 = 2.00941 (limit 2)


n= 1000 reps=   8   m2 = 1.00039 (exact 1.00100)   m4 = 2.00170 (limit 2)
